In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import random
from dataclasses import dataclass

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Adat gyökér
root_path = "/content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train"

# Ide mentünk mindent az UNet FNO-hoz
output_root = "/content/drive/MyDrive/Brain MRI/Unet_FNO_2"

@dataclass
class Config:
    # seedek
    seed_global: int = 42
    seed_split: int = 123

    # tréning
    batch_size: int = 4
    num_epochs: int = 100
    lr: float = 1e-3
    weight_decay: float = 1e-5
    num_workers: int = 2
    val_ratio: float = 0.2
    evals_per_epoch: int = 4

    # warmup + scheduler
    use_warmup: bool = True
    warmup_epochs: int = 5
    warmup_start_lr: float = 1e-5
    scheduler_type: str = "cosine"  # "cosine" vagy "none"
    eta_min: float = 1e-5           # cosine min lr

    # képméret, osztályok
    image_height: int = 256
    image_width: int = 256
    num_classes: int = 2

    # loss
    loss_bce_weight: float = 0.5
    loss_dice_weight: float = 0.5
    loss_smooth: float = 1.0

    # pathok
    model_dir: str = os.path.join(output_root, "models")
    plots_dir: str = os.path.join(output_root, "plots")
    preds_dir: str = os.path.join(output_root, "predictions")
    image_dir: str = os.path.join(root_path, "images")
    mask_dir: str = os.path.join(root_path, "masks")

cfg = Config()

print("Images dir:", cfg.image_dir, "exists:", os.path.isdir(cfg.image_dir))
print("Masks dir:", cfg.mask_dir, "exists:", os.path.isdir(cfg.mask_dir))

os.makedirs(cfg.model_dir, exist_ok=True)
os.makedirs(cfg.plots_dir, exist_ok=True)
os.makedirs(cfg.preds_dir, exist_ok=True)


Device: cuda
Images dir: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/images exists: True
Masks dir: /content/drive/MyDrive/Brain MRI/brisc2025/segmentation_task/train/masks exists: True


In [4]:
def set_seed(seed: int):
    print(f"*** Setting global seed to {seed} ***")
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed_global)


*** Setting global seed to 42 ***


In [5]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir: str, mask_dir: str):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])
        self.mask_files = sorted([
            f for f in os.listdir(mask_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))
        ])

        if len(self.image_files) == 0:
            raise ValueError(f"Nincs kép a(z) {image_dir} mappában!")
        if len(self.mask_files) == 0:
            raise ValueError(f"Nincs maszk a(z) {mask_dir} mappában!")
        assert len(self.image_files) == len(self.mask_files), \
            "A képek és maszkok száma nem egyezik!"

        print(f"Találtam {len(self.image_files)} képet és maszkot.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx: int):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        image = image.resize((cfg.image_width, cfg.image_height))
        mask = mask.resize((cfg.image_width, cfg.image_height), Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)  # 0 = background, 1 = tumor

        image = np.expand_dims(image, axis=0)  # (1, H, W)

        return torch.from_numpy(image), torch.from_numpy(mask)


In [6]:
full_dataset = SegmentationDataset(cfg.image_dir, cfg.mask_dir)
dataset_size = len(full_dataset)
val_size = int(dataset_size * cfg.val_ratio)
train_size = dataset_size - val_size

print("Összes minta:", dataset_size)
print("Train:", train_size, "Val:", val_size)

set_seed(cfg.seed_split)
generator = torch.Generator().manual_seed(cfg.seed_split)

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

images, masks = next(iter(train_loader))
print("Batch images:", images.shape)
print("Batch masks:", masks.shape)


Találtam 3933 képet és maszkot.
Összes minta: 3933
Train: 3147 Val: 786
*** Setting global seed to 123 ***
Batch images: torch.Size([4, 1, 256, 256])
Batch masks: torch.Size([4, 256, 256])


In [7]:
@torch.jit.script
def compl_mul2d(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    # (batch, in_channel, x, y), (in_channel, out_channel, x, y) -> (batch, out_channel, x, y)
    return torch.einsum("bixy, ioxy->boxy", a, b)

class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat)
        )

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfftn(x, dim=[2, 3])

        out_ft = torch.zeros(
            batchsize,
            self.out_channels,
            x.size(-2),
            x.size(-1) // 2 + 1,
            device=x.device,
            dtype=torch.cfloat
        )

        out_ft[:, :, :self.modes1, :self.modes2] = \
            compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

        x = torch.fft.irfftn(out_ft, s=(x.size(-2), x.size(-1)), dim=[2, 3])
        return x

class FourierLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, modes=4):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=True
        )
        self.conv_fno = SpectralConv2d(in_channels, out_channels, modes, modes)

    def forward(self, x):
        x1 = self.conv(x)
        x2 = self.conv_fno(x)
        return x1 + x2

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, modes=4):
        super().__init__()
        self.net = nn.Sequential(
            FourierLayer(in_ch, out_ch, kernel_size=3, stride=1, padding=1, modes=modes),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            FourierLayer(out_ch, out_ch, kernel_size=3, stride=1, padding=1, modes=modes),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNetFNO(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, modes=4):
        super().__init__()
        # Encoder
        self.down1 = DoubleConv(in_channels, 64, modes=modes)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(64, 128, modes=modes)
        self.pool2 = nn.MaxPool2d(2)

        # Bridge
        self.bridge = DoubleConv(128, 256, modes=modes)

        # Decoder
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128, modes=modes)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64, modes=modes)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # encoder
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        # bridge
        b = self.bridge(p2)

        # decoder
        u2 = self.up2(b)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        out = self.out_conv(u1)
        return out

# gyors teszt
model_test = UNetFNO(in_channels=1, out_channels=cfg.num_classes, modes=4).to(device)
x_test = torch.randn(4, 1, cfg.image_height, cfg.image_width).to(device)
y_test = model_test(x_test)
print("Model output shape:", y_test.shape)
print("Total params:", sum(p.numel() for p in model_test.parameters()))


Model output shape: torch.Size([4, 2, 256, 256])
Total params: 7895682


In [8]:
class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.ce_loss = nn.CrossEntropyLoss()

    def dice_loss(self, logits, targets, num_classes=2):
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        dice_scores = []
        for cls in range(num_classes):
            pred_cls = probs[:, cls, :, :]
            target_cls = targets_one_hot[:, cls, :, :]
            intersection = (pred_cls * target_cls).sum()
            union = pred_cls.sum() + target_cls.sum()
            dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)

        mean_dice = sum(dice_scores) / len(dice_scores)
        return 1.0 - mean_dice

    def forward(self, logits, targets):
        ce_loss = self.ce_loss(logits, targets)
        dice_loss = self.dice_loss(logits, targets, num_classes=logits.size(1))
        return self.bce_weight * ce_loss + self.dice_weight * dice_loss


In [9]:
def dice_score(logits, targets, num_classes=2):
    preds = torch.argmax(logits, dim=1)
    batch_size = preds.size(0)
    sample_dice_scores = []

    for i in range(batch_size):
        dice_scores = []
        for cls in range(num_classes):
            pred_mask = (preds[i] == cls).float()
            target_mask = (targets[i] == cls).float()
            intersection = (pred_mask * target_mask).sum()
            dice = (2.0 * intersection) / (pred_mask.sum() + target_mask.sum() + 1e-8)
            dice_scores.append(dice.item())
        sample_dice_scores.append(sum(dice_scores) / len(dice_scores))

    return sample_dice_scores

def compute_iou(logits, targets, num_classes=2):
    preds = torch.argmax(logits, dim=1)
    batch_size = preds.size(0)
    sample_iou_scores = []

    for i in range(batch_size):
        ious = []
        for cls in range(num_classes):
            pred_inds = (preds[i] == cls)
            target_inds = (targets[i] == cls)
            intersection = (pred_inds & target_inds).sum().item()
            union = (pred_inds | target_inds).sum().item()
            if union == 0:
                continue
            ious.append(intersection / union)
        if len(ious) == 0:
            sample_iou_scores.append(0.0)
        else:
            sample_iou_scores.append(sum(ious) / len(ious))

    return sample_iou_scores


In [10]:
def get_lr_for_epoch(epoch, base_lr, cfg: Config):
    # epoch: 0-indexelt
    if cfg.use_warmup and epoch < cfg.warmup_epochs:
        alpha = epoch / max(1, cfg.warmup_epochs)
        return cfg.warmup_start_lr + alpha * (base_lr - cfg.warmup_start_lr)
    return base_lr

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_dice = []
    all_iou = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        running_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

        dice_scores = dice_score(outputs, masks, num_classes=cfg.num_classes)
        iou_scores = compute_iou(outputs, masks, num_classes=cfg.num_classes)

        all_dice.extend(dice_scores)
        all_iou.extend(iou_scores)

    val_loss = running_loss / total_samples
    val_dice = sum(all_dice) / len(all_dice)
    val_iou = sum(all_iou) / len(all_iou)
    return val_loss, val_dice, val_iou

def train_model_with_logging(run_name: str, seed: int):
    set_seed(seed)

    model = UNetFNO(
        in_channels=1,
        out_channels=cfg.num_classes,
        modes=4
    ).to(device)

    print(f"Using UNetFNO model (run: {run_name}, seed: {seed})")
    print("Total params:", sum(p.numel() for p in model.parameters()))

    criterion = CombinedBCEDiceLoss(
        bce_weight=cfg.loss_bce_weight,
        dice_weight=cfg.loss_dice_weight,
        smooth=cfg.loss_smooth
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    batches_per_epoch = len(train_loader)

    if cfg.scheduler_type == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg.num_epochs,
            eta_min=cfg.eta_min
        )
        print(f"Using CosineAnnealingLR: T_max={cfg.num_epochs}, eta_min={cfg.eta_min}")
    else:
        scheduler = None
        print("No scheduler, constant lr (with warmup)")

    print(f"Evaluations per epoch: {cfg.evals_per_epoch}")
    eval_interval = max(1, batches_per_epoch // cfg.evals_per_epoch)
    print(f"Evaluation interval: every {eval_interval} batches")

    train_history = []
    val_history = []
    best_val_dice = 0.0

    for epoch in range(1, cfg.num_epochs + 1):
        model.train()
        running_loss = 0.0
        total_samples = 0
        all_dice_scores = []
        batch_count = 0

        # epoch-1, mert get_lr_for_epoch 0-indexelt
        epoch_lr = get_lr_for_epoch(epoch - 1, cfg.lr, cfg)
        for param_group in optimizer.param_groups:
            param_group["lr"] = epoch_lr

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            dice_scores = dice_score(outputs.detach(), masks, num_classes=cfg.num_classes)
            all_dice_scores.extend(dice_scores)

            running_loss += loss.item() * images.size(0)
            total_samples += images.size(0)
            batch_count += 1

            if batch_count % eval_interval == 0 or batch_count == batches_per_epoch:
                train_loss = running_loss / total_samples
                train_dice = sum(all_dice_scores) / len(all_dice_scores)

                val_loss, val_dice, val_iou = validate(model, val_loader, criterion)

                current_lr = optimizer.param_groups[0]["lr"]
                step_in_epoch = batch_count / batches_per_epoch

                print(
                    f"Epoch {epoch:02d} step {step_in_epoch: .2f} | "
                    f"lr={current_lr:.6f} | "
                    f"train_loss={train_loss:.4f} | train_dice={train_dice:.4f} | "
                    f"val_loss={val_loss:.4f} | val_dice={val_dice:.4f} | val_iou={val_iou:.4f}"
                )

                train_history.append((train_loss, train_dice))
                val_history.append((val_loss, val_dice, val_iou))

                if val_dice > best_val_dice:
                    best_val_dice = val_dice
                    save_path = os.path.join(cfg.model_dir, f"{run_name}_best_model.pt")
                    torch.save({
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": val_dice,
                        "val_iou": val_iou,
                        "seed": seed,
                    }, save_path)
                    print(f"Új legjobb modell mentve: epoch {epoch}, Dice={val_dice:.4f}, IoU={val_iou:.4f}")

        if scheduler is not None:
            scheduler.step()

    return {
        "run_name": run_name,
        "seed": seed,
        "train_history": train_history,
        "val_history": val_history,
        "best_val_dice": best_val_dice
    }


In [ ]:
seeds = [2021, 2022, 2023, 2024, 2025]
all_results = []

for i, s in enumerate(seeds, start=1):
    print("=" * 60)
    print(f"RUN {i} / 5 (seed={s})")
    result = train_model_with_logging(run_name=f"unetfno_run{i}", seed=s)
    print("Best val Dice:", result["best_val_dice"])
    all_results.append(result)


RUN 1 / 5 (seed=2021)
*** Setting global seed to 2021 ***
Using UNetFNO model (run: unetfno_run1, seed: 2021)
Total params: 7895682
Using CosineAnnealingLR: T_max=100, eta_min=1e-05
Evaluations per epoch: 4
Evaluation interval: every 196 batches
Epoch 01 step  0.25 | lr=0.000010 | train_loss=0.5315 | train_dice=0.5472 | val_loss=0.4704 | val_dice=0.6304 | val_iou=0.5734
Új legjobb modell mentve: epoch 1, Dice=0.6304, IoU=0.5734
Epoch 01 step  0.50 | lr=0.000010 | train_loss=0.3628 | train_dice=0.6179 | val_loss=0.1451 | val_dice=0.7520 | val_iou=0.7023
Új legjobb modell mentve: epoch 1, Dice=0.7520, IoU=0.7023
Epoch 01 step  0.75 | lr=0.000010 | train_loss=0.2860 | train_dice=0.6733 | val_loss=0.1330 | val_dice=0.7858 | val_iou=0.7294
Új legjobb modell mentve: epoch 1, Dice=0.7858, IoU=0.7294
Epoch 01 step  1.00 | lr=0.000010 | train_loss=0.2464 | train_dice=0.7017 | val_loss=0.1176 | val_dice=0.7995 | val_iou=0.7491
Új legjobb modell mentve: epoch 1, Dice=0.7995, IoU=0.7491
Epoch 01 s

In [ ]:
import numpy as np

result_unetfno = all_results[0]  # pl. az első futás

train_losses = np.array(result_unetfno["train_history"])[:, 0]
train_dices = np.array(result_unetfno["train_history"])[:, 1]
val_losses = np.array(result_unetfno["val_history"])[:, 0]
val_dices = np.array(result_unetfno["val_history"])[:, 1]
val_ious = np.array(result_unetfno["val_history"])[:, 2]

epochs = np.arange(1, len(train_losses) + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs, train_losses, label="Train loss")
plt.plot(epochs, val_losses, label="Val loss")
plt.legend()
plt.grid(True)
plt.xlabel("Eval step")
plt.ylabel("Loss")
plt.tight_layout()
plt.savefig(os.path.join(cfg.plots_dir, "unetfno_loss.png"))
plt.close()

plt.figure(figsize=(6, 4))
plt.plot(epochs, val_dices, label="Val Dice")
plt.plot(epochs, val_ious, label="Val IoU")
plt.legend()
plt.grid(True)
plt.xlabel("Eval step")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig(os.path.join(cfg.plots_dir, "unetfno_metrics.png"))
plt.close()


In [ ]:
# Válassz egy run-t, pl. unetfno_run1
best_model_path = os.path.join(cfg.model_dir, "unetfno_run1_best_model.pt")
print("Best model path:", best_model_path)

loaded_model = UNetFNO(
    in_channels=1,
    out_channels=cfg.num_classes,
    modes=4
).to(device)

checkpoint = torch.load(best_model_path, map_location=device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

print("Best model loaded successfully.")
print(f"Epoch: {checkpoint['epoch']}, Val IoU: {checkpoint['val_iou']:.4f}, Val Dice: {checkpoint['val_dice']:.4f}")


In [ ]:
def visualize_and_save_predictions(model, loader, num_samples=4, save_prefix="unetfno"):
    model.eval()
    images, masks = next(iter(loader))
    images = images.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

    images_np = images.cpu().numpy()
    masks_np = masks.cpu().numpy()
    preds_np = preds.cpu().numpy()

    plt.figure(figsize=(10, num_samples * 3))
    for i in range(num_samples):
        # Input
        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(images_np[i][0], cmap="gray")
        plt.title("Input")
        plt.axis("off")

        # Ground truth (binary, fekete-fehér)
        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(masks_np[i], cmap="gray")
        plt.title("GT mask")
        plt.axis("off")

        # Prediction (binary, fekete-fehér)
        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(preds_np[i], cmap="gray")
        plt.title("Pred mask")
        plt.axis("off")

    plt.tight_layout()
    out_path = os.path.join(cfg.preds_dir, f"{save_prefix}_predictions.png")
    plt.savefig(out_path)
    plt.close()
    print("Predictions saved to:", out_path)

visualize_and_save_predictions(loaded_model, val_loader, num_samples=4, save_prefix="unetfno_run1")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Válassz egy run-t
result_unetfno = all_results[1]

# History-k kinyerése
train_hist = np.array(result_unetfno["train_history"])
val_hist = np.array(result_unetfno["val_history"])

train_losses = train_hist[:, 0]
train_dices  = train_hist[:, 1]

val_losses = val_hist[:, 0]
val_dices  = val_hist[:, 1]
val_ious   = val_hist[:, 2]

epochs = np.arange(1, len(train_losses) + 1)

# 3 paneles ábra
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -----------------------------
# 1. Training and Validation Loss
# -----------------------------
axes[0].plot(epochs, train_losses, label="Train Loss")
axes[0].plot(epochs, val_losses, label="Val Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# -----------------------------
# 2. Dice Score
# -----------------------------
axes[1].plot(epochs, train_dices, label="Train Dice")
axes[1].plot(epochs, val_dices, label="Val Dice")
axes[1].set_title("Dice Score")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# -----------------------------
# 3. Validation Metrics
# -----------------------------
axes[2].plot(epochs, val_dices, label="Val Dice")
axes[2].plot(epochs, val_ious, label="Val IoU")
axes[2].set_title("Validation Metrics")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()

# Mentés
save_path = os.path.join(cfg.plots_dir, "unetfno_summary_plot.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Plot elmentve ide: {save_path}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import numpy as np

# 1. Ide másold be a teljes log szöveget
log_text = """
Epoch 01 step  0.25 | lr=0.000010 | train_loss=0.5315 | train_dice=0.5472 | val_loss=0.4704 | val_dice=0.6304 | val_iou=0.5734
Új legjobb modell mentve: epoch 1, Dice=0.6304, IoU=0.5734
Epoch 01 step  0.50 | lr=0.000010 | train_loss=0.3628 | train_dice=0.6179 | val_loss=0.1451 | val_dice=0.7520 | val_iou=0.7023
Új legjobb modell mentve: epoch 1, Dice=0.7520, IoU=0.7023
Epoch 01 step  0.75 | lr=0.000010 | train_loss=0.2860 | train_dice=0.6733 | val_loss=0.1329 | val_dice=0.7858 | val_iou=0.7295
Új legjobb modell mentve: epoch 1, Dice=0.7858, IoU=0.7295
Epoch 01 step  1.00 | lr=0.000010 | train_loss=0.2464 | train_dice=0.7017 | val_loss=0.1176 | val_dice=0.7995 | val_iou=0.7491
Új legjobb modell mentve: epoch 1, Dice=0.7995, IoU=0.7491
Epoch 01 step  1.00 | lr=0.000010 | train_loss=0.2460 | train_dice=0.7018 | val_loss=0.1171 | val_dice=0.8012 | val_iou=0.7503
Új legjobb modell mentve: epoch 1, Dice=0.8012, IoU=0.7503
Epoch 02 step  0.25 | lr=0.000208 | train_loss=0.3033 | train_dice=0.7452 | val_loss=0.2242 | val_dice=0.7793 | val_iou=0.7208
Epoch 02 step  0.50 | lr=0.000208 | train_loss=0.2284 | train_dice=0.7458 | val_loss=0.1334 | val_dice=0.7845 | val_iou=0.7289
Epoch 02 step  0.75 | lr=0.000208 | train_loss=0.1928 | train_dice=0.7605 | val_loss=0.1257 | val_dice=0.7738 | val_iou=0.7276
Epoch 02 step  1.00 | lr=0.000208 | train_loss=0.1723 | train_dice=0.7719 | val_loss=0.1071 | val_dice=0.8124 | val_iou=0.7608
Új legjobb modell mentve: epoch 2, Dice=0.8124, IoU=0.7608
Epoch 02 step  1.00 | lr=0.000208 | train_loss=0.1720 | train_dice=0.7722 | val_loss=0.1062 | val_dice=0.8120 | val_iou=0.7636
Epoch 03 step  0.25 | lr=0.000406 | train_loss=0.1581 | train_dice=0.7927 | val_loss=0.1187 | val_dice=0.8106 | val_iou=0.7616
Epoch 03 step  0.50 | lr=0.000406 | train_loss=0.1366 | train_dice=0.7962 | val_loss=0.1106 | val_dice=0.8094 | val_iou=0.7626
Epoch 03 step  0.75 | lr=0.000406 | train_loss=0.1284 | train_dice=0.7972 | val_loss=0.1059 | val_dice=0.8191 | val_iou=0.7727
Új legjobb modell mentve: epoch 3, Dice=0.8191, IoU=0.7727
Epoch 03 step  1.00 | lr=0.000406 | train_loss=0.1221 | train_dice=0.8034 | val_loss=0.0979 | val_dice=0.8238 | val_iou=0.7807
Új legjobb modell mentve: epoch 3, Dice=0.8238, IoU=0.7807
Epoch 03 step  1.00 | lr=0.000406 | train_loss=0.1219 | train_dice=0.8039 | val_loss=0.0976 | val_dice=0.8256 | val_iou=0.7818
Új legjobb modell mentve: epoch 3, Dice=0.8256, IoU=0.7818
Epoch 04 step  0.25 | lr=0.000604 | train_loss=0.1087 | train_dice=0.8138 | val_loss=0.1051 | val_dice=0.8232 | val_iou=0.7777
Epoch 04 step  0.50 | lr=0.000604 | train_loss=0.1020 | train_dice=0.8186 | val_loss=0.1084 | val_dice=0.8183 | val_iou=0.7681
Epoch 04 step  0.75 | lr=0.000604 | train_loss=0.1020 | train_dice=0.8207 | val_loss=0.1026 | val_dice=0.8260 | val_iou=0.7776
Új legjobb modell mentve: epoch 4, Dice=0.8260, IoU=0.7776
Epoch 04 step  1.00 | lr=0.000604 | train_loss=0.0987 | train_dice=0.8265 | val_loss=0.0943 | val_dice=0.8289 | val_iou=0.7871
Új legjobb modell mentve: epoch 4, Dice=0.8289, IoU=0.7871
Epoch 04 step  1.00 | lr=0.000604 | train_loss=0.0987 | train_dice=0.8264 | val_loss=0.0951 | val_dice=0.8303 | val_iou=0.7881
Új legjobb modell mentve: epoch 4, Dice=0.8303, IoU=0.7881
Epoch 05 step  0.25 | lr=0.000802 | train_loss=0.1001 | train_dice=0.8242 | val_loss=0.0951 | val_dice=0.8372 | val_iou=0.7894
Új legjobb modell mentve: epoch 5, Dice=0.8372, IoU=0.7894
Epoch 05 step  0.50 | lr=0.000802 | train_loss=0.0996 | train_dice=0.8252 | val_loss=0.0948 | val_dice=0.8314 | val_iou=0.7893
Epoch 05 step  0.75 | lr=0.000802 | train_loss=0.0943 | train_dice=0.8339 | val_loss=0.1003 | val_dice=0.8368 | val_iou=0.7818
Epoch 05 step  1.00 | lr=0.000802 | train_loss=0.0926 | train_dice=0.8380 | val_loss=0.1039 | val_dice=0.8379 | val_iou=0.7821
Új legjobb modell mentve: epoch 5, Dice=0.8379, IoU=0.7821
Epoch 05 step  1.00 | lr=0.000802 | train_loss=0.0926 | train_dice=0.8382 | val_loss=0.0964 | val_dice=0.8488 | val_iou=0.7970
Új legjobb modell mentve: epoch 5, Dice=0.8488, IoU=0.7970
Epoch 06 step  0.25 | lr=0.001000 | train_loss=0.0906 | train_dice=0.8401 | val_loss=0.0893 | val_dice=0.8455 | val_iou=0.8016
Epoch 06 step  0.50 | lr=0.001000 | train_loss=0.0883 | train_dice=0.8449 | val_loss=0.0867 | val_dice=0.8545 | val_iou=0.8111
Új legjobb modell mentve: epoch 6, Dice=0.8545, IoU=0.8111
Epoch 06 step  0.75 | lr=0.001000 | train_loss=0.0842 | train_dice=0.8517 | val_loss=0.0815 | val_dice=0.8644 | val_iou=0.8182
Új legjobb modell mentve: epoch 6, Dice=0.8644, IoU=0.8182
Epoch 06 step  1.00 | lr=0.001000 | train_loss=0.0839 | train_dice=0.8520 | val_loss=0.0893 | val_dice=0.8441 | val_iou=0.8013
Epoch 06 step  1.00 | lr=0.001000 | train_loss=0.0838 | train_dice=0.8521 | val_loss=0.0879 | val_dice=0.8477 | val_iou=0.8052
Epoch 07 step  0.25 | lr=0.001000 | train_loss=0.0844 | train_dice=0.8558 | val_loss=0.0837 | val_dice=0.8547 | val_iou=0.8086
Epoch 07 step  0.50 | lr=0.001000 | train_loss=0.0794 | train_dice=0.8592 | val_loss=0.0806 | val_dice=0.8596 | val_iou=0.8171
Epoch 07 step  0.75 | lr=0.001000 | train_loss=0.0776 | train_dice=0.8623 | val_loss=0.0822 | val_dice=0.8588 | val_iou=0.8176
Epoch 07 step  1.00 | lr=0.001000 | train_loss=0.0762 | train_dice=0.8657 | val_loss=0.0749 | val_dice=0.8741 | val_iou=0.8325
Új legjobb modell mentve: epoch 7, Dice=0.8741, IoU=0.8325
Epoch 07 step  1.00 | lr=0.001000 | train_loss=0.0760 | train_dice=0.8660 | val_loss=0.0752 | val_dice=0.8730 | val_iou=0.8310
Epoch 08 step  0.25 | lr=0.001000 | train_loss=0.0641 | train_dice=0.8907 | val_loss=0.0756 | val_dice=0.8702 | val_iou=0.8289
Epoch 08 step  0.50 | lr=0.001000 | train_loss=0.0667 | train_dice=0.8854 | val_loss=0.0780 | val_dice=0.8692 | val_iou=0.8198
Epoch 08 step  0.75 | lr=0.001000 | train_loss=0.0685 | train_dice=0.8803 | val_loss=0.0883 | val_dice=0.8443 | val_iou=0.8034
Epoch 08 step  1.00 | lr=0.001000 | train_loss=0.0681 | train_dice=0.8793 | val_loss=0.0739 | val_dice=0.8788 | val_iou=0.8359
Új legjobb modell mentve: epoch 8, Dice=0.8788, IoU=0.8359
Epoch 08 step  1.00 | lr=0.001000 | train_loss=0.0681 | train_dice=0.8791 | val_loss=0.0744 | val_dice=0.8802 | val_iou=0.8360
Új legjobb modell mentve: epoch 8, Dice=0.8802, IoU=0.8360
Epoch 09 step  0.25 | lr=0.001000 | train_loss=0.0637 | train_dice=0.8916 | val_loss=0.0727 | val_dice=0.8749 | val_iou=0.8320
Epoch 09 step  0.50 | lr=0.001000 | train_loss=0.0617 | train_dice=0.8912 | val_loss=0.0683 | val_dice=0.8848 | val_iou=0.8426
Új legjobb modell mentve: epoch 9, Dice=0.8848, IoU=0.8426
Epoch 09 step  0.75 | lr=0.001000 | train_loss=0.0629 | train_dice=0.8898 | val_loss=0.0789 | val_dice=0.8662 | val_iou=0.8256
Epoch 09 step  1.00 | lr=0.001000 | train_loss=0.0627 | train_dice=0.8893 | val_loss=0.0764 | val_dice=0.8734 | val_iou=0.8246
Epoch 09 step  1.00 | lr=0.001000 | train_loss=0.0627 | train_dice=0.8891 | val_loss=0.0771 | val_dice=0.8731 | val_iou=0.8244
Epoch 10 step  0.25 | lr=0.001000 | train_loss=0.0603 | train_dice=0.8946 | val_loss=0.0704 | val_dice=0.8824 | val_iou=0.8350
Epoch 10 step  0.50 | lr=0.001000 | train_loss=0.0625 | train_dice=0.8891 | val_loss=0.0678 | val_dice=0.8850 | val_iou=0.8432
Új legjobb modell mentve: epoch 10, Dice=0.8850, IoU=0.8432
Epoch 10 step  0.75 | lr=0.001000 | train_loss=0.0615 | train_dice=0.8899 | val_loss=0.0661 | val_dice=0.8886 | val_iou=0.8456
Új legjobb modell mentve: epoch 10, Dice=0.8886, IoU=0.8456
Epoch 10 step  1.00 | lr=0.001000 | train_loss=0.0616 | train_dice=0.8908 | val_loss=0.0804 | val_dice=0.8750 | val_iou=0.8301
Epoch 10 step  1.00 | lr=0.001000 | train_loss=0.0616 | train_dice=0.8910 | val_loss=0.0772 | val_dice=0.8787 | val_iou=0.8347
Epoch 11 step  0.25 | lr=0.001000 | train_loss=0.0594 | train_dice=0.8935 | val_loss=0.0671 | val_dice=0.8900 | val_iou=0.8477
Új legjobb modell mentve: epoch 11, Dice=0.8900, IoU=0.8477
Epoch 11 step  0.50 | lr=0.001000 | train_loss=0.0580 | train_dice=0.8946 | val_loss=0.0717 | val_dice=0.8730 | val_iou=0.8299
Epoch 11 step  0.75 | lr=0.001000 | train_loss=0.0596 | train_dice=0.8942 | val_loss=0.0678 | val_dice=0.8800 | val_iou=0.8420
Epoch 11 step  1.00 | lr=0.001000 | train_loss=0.0591 | train_dice=0.8950 | val_loss=0.0673 | val_dice=0.8880 | val_iou=0.8463
Epoch 11 step  1.00 | lr=0.001000 | train_loss=0.0591 | train_dice=0.8949 | val_loss=0.0675 | val_dice=0.8844 | val_iou=0.8441
Epoch 12 step  0.25 | lr=0.001000 | train_loss=0.0530 | train_dice=0.9108 | val_loss=0.0616 | val_dice=0.8923 | val_iou=0.8530
Új legjobb modell mentve: epoch 12, Dice=0.8923, IoU=0.8530
Epoch 12 step  0.50 | lr=0.001000 | train_loss=0.0534 | train_dice=0.9063 | val_loss=0.0646 | val_dice=0.8902 | val_iou=0.8506
Epoch 12 step  0.75 | lr=0.001000 | train_loss=0.0524 | train_dice=0.9064 | val_loss=0.0732 | val_dice=0.8857 | val_iou=0.8363
Epoch 12 step  1.00 | lr=0.001000 | train_loss=0.0534 | train_dice=0.9047 | val_loss=0.0608 | val_dice=0.8998 | val_iou=0.8564
Új legjobb modell mentve: epoch 12, Dice=0.8998, IoU=0.8564
Epoch 12 step  1.00 | lr=0.001000 | train_loss=0.0535 | train_dice=0.9047 | val_loss=0.0601 | val_dice=0.8994 | val_iou=0.8569
Epoch 13 step  0.25 | lr=0.001000 | train_loss=0.0474 | train_dice=0.9138 | val_loss=0.0601 | val_dice=0.9012 | val_iou=0.8599
Új legjobb modell mentve: epoch 13, Dice=0.9012, IoU=0.8599
Epoch 13 step  0.50 | lr=0.001000 | train_loss=0.0493 | train_dice=0.9127 | val_loss=0.0597 | val_dice=0.8935 | val_iou=0.8542
Epoch 13 step  0.75 | lr=0.001000 | train_loss=0.0517 | train_dice=0.9092 | val_loss=0.0603 | val_dice=0.8956 | val_iou=0.8577
Epoch 13 step  1.00 | lr=0.001000 | train_loss=0.0503 | train_dice=0.9110 | val_loss=0.0656 | val_dice=0.8875 | val_iou=0.8485
Epoch 13 step  1.00 | lr=0.001000 | train_loss=0.0503 | train_dice=0.9109 | val_loss=0.0649 | val_dice=0.8865 | val_iou=0.8488
Epoch 14 step  0.25 | lr=0.001000 | train_loss=0.0501 | train_dice=0.9121 | val_loss=0.0564 | val_dice=0.9037 | val_iou=0.8630
Új legjobb modell mentve: epoch 14, Dice=0.9037, IoU=0.8630
Epoch 14 step  0.50 | lr=0.001000 | train_loss=0.0492 | train_dice=0.9125 | val_loss=0.0681 | val_dice=0.8843 | val_iou=0.8470
Epoch 14 step  0.75 | lr=0.001000 | train_loss=0.0491 | train_dice=0.9128 | val_loss=0.0590 | val_dice=0.9044 | val_iou=0.8635
Új legjobb modell mentve: epoch 14, Dice=0.9044, IoU=0.8635
Epoch 14 step  1.00 | lr=0.001000 | train_loss=0.0481 | train_dice=0.9134 | val_loss=0.0603 | val_dice=0.9016 | val_iou=0.8604
Epoch 14 step  1.00 | lr=0.001000 | train_loss=0.0481 | train_dice=0.9135 | val_loss=0.0601 | val_dice=0.9026 | val_iou=0.8621
Epoch 15 step  0.25 | lr=0.001000 | train_loss=0.0414 | train_dice=0.9234 | val_loss=0.0540 | val_dice=0.9064 | val_iou=0.8672
Új legjobb modell mentve: epoch 15, Dice=0.9064, IoU=0.8672
Epoch 15 step  0.50 | lr=0.001000 | train_loss=0.0432 | train_dice=0.9210 | val_loss=0.0617 | val_dice=0.8933 | val_iou=0.8532
Epoch 15 step  0.75 | lr=0.001000 | train_loss=0.0448 | train_dice=0.9183 | val_loss=0.0625 | val_dice=0.8922 | val_iou=0.8526
Epoch 15 step  1.00 | lr=0.001000 | train_loss=0.0453 | train_dice=0.9174 | val_loss=0.0657 | val_dice=0.8901 | val_iou=0.8479
Epoch 15 step  1.00 | lr=0.001000 | train_loss=0.0453 | train_dice=0.9176 | val_loss=0.0652 | val_dice=0.8873 | val_iou=0.8457
Epoch 16 step  0.25 | lr=0.001000 | train_loss=0.0492 | train_dice=0.9106 | val_loss=0.0578 | val_dice=0.8992 | val_iou=0.8592
Epoch 16 step  0.50 | lr=0.001000 | train_loss=0.0456 | train_dice=0.9174 | val_loss=0.0599 | val_dice=0.9012 | val_iou=0.8604
Epoch 16 step  0.75 | lr=0.001000 | train_loss=0.0436 | train_dice=0.9200 | val_loss=0.0633 | val_dice=0.8918 | val_iou=0.8532
Epoch 16 step  1.00 | lr=0.001000 | train_loss=0.0437 | train_dice=0.9194 | val_loss=0.0566 | val_dice=0.9022 | val_iou=0.8629
Epoch 16 step  1.00 | lr=0.001000 | train_loss=0.0438 | train_dice=0.9195 | val_loss=0.0555 | val_dice=0.9049 | val_iou=0.8652
Epoch 17 step  0.25 | lr=0.001000 | train_loss=0.0367 | train_dice=0.9305 | val_loss=0.0558 | val_dice=0.9060 | val_iou=0.8660
Epoch 17 step  0.50 | lr=0.001000 | train_loss=0.0405 | train_dice=0.9268 | val_loss=0.0548 | val_dice=0.9044 | val_iou=0.8647
Epoch 17 step  0.75 | lr=0.001000 | train_loss=0.0402 | train_dice=0.9256 | val_loss=0.0564 | val_dice=0.9029 | val_iou=0.8625
Epoch 17 step  1.00 | lr=0.001000 | train_loss=0.0414 | train_dice=0.9242 | val_loss=0.0578 | val_dice=0.9053 | val_iou=0.8641
Epoch 17 step  1.00 | lr=0.001000 | train_loss=0.0414 | train_dice=0.9242 | val_loss=0.0575 | val_dice=0.9068 | val_iou=0.8660
Új legjobb modell mentve: epoch 17, Dice=0.9068, IoU=0.8660
Epoch 18 step  0.25 | lr=0.001000 | train_loss=0.0398 | train_dice=0.9294 | val_loss=0.0565 | val_dice=0.9013 | val_iou=0.8638
Epoch 18 step  0.50 | lr=0.001000 | train_loss=0.0426 | train_dice=0.9237 | val_loss=0.0583 | val_dice=0.9048 | val_iou=0.8650
Epoch 18 step  0.75 | lr=0.001000 | train_loss=0.0416 | train_dice=0.9260 | val_loss=0.0589 | val_dice=0.9064 | val_iou=0.8636
Epoch 18 step  1.00 | lr=0.001000 | train_loss=0.0411 | train_dice=0.9263 | val_loss=0.0538 | val_dice=0.9073 | val_iou=0.8696
Új legjobb modell mentve: epoch 18, Dice=0.9073, IoU=0.8696
Epoch 18 step  1.00 | lr=0.001000 | train_loss=0.0411 | train_dice=0.9260 | val_loss=0.0542 | val_dice=0.9068 | val_iou=0.8688
Epoch 19 step  0.25 | lr=0.001000 | train_loss=0.0380 | train_dice=0.9281 | val_loss=0.0559 | val_dice=0.9065 | val_iou=0.8667
Epoch 19 step  0.50 | lr=0.001000 | train_loss=0.0374 | train_dice=0.9301 | val_loss=0.0568 | val_dice=0.9085 | val_iou=0.8683
Új legjobb modell mentve: epoch 19, Dice=0.9085, IoU=0.8683
Epoch 19 step  0.75 | lr=0.001000 | train_loss=0.0371 | train_dice=0.9311 | val_loss=0.0537 | val_dice=0.9096 | val_iou=0.8695
Új legjobb modell mentve: epoch 19, Dice=0.9096, IoU=0.8695
Epoch 19 step  1.00 | lr=0.001000 | train_loss=0.0383 | train_dice=0.9298 | val_loss=0.0588 | val_dice=0.8985 | val_iou=0.8594
Epoch 19 step  1.00 | lr=0.001000 | train_loss=0.0383 | train_dice=0.9298 | val_loss=0.0579 | val_dice=0.9003 | val_iou=0.8619
Epoch 20 step  0.25 | lr=0.001000 | train_loss=0.0381 | train_dice=0.9335 | val_loss=0.0556 | val_dice=0.9054 | val_iou=0.8660
Epoch 20 step  0.50 | lr=0.001000 | train_loss=0.0376 | train_dice=0.9329 | val_loss=0.0536 | val_dice=0.9111 | val_iou=0.8713
Új legjobb modell mentve: epoch 20, Dice=0.9111, IoU=0.8713
Epoch 20 step  0.75 | lr=0.001000 | train_loss=0.0379 | train_dice=0.9317 | val_loss=0.0574 | val_dice=0.9081 | val_iou=0.8675
Epoch 20 step  1.00 | lr=0.001000 | train_loss=0.0381 | train_dice=0.9311 | val_loss=0.0576 | val_dice=0.9031 | val_iou=0.8619
Epoch 20 step  1.00 | lr=0.001000 | train_loss=0.0380 | train_dice=0.9312 | val_loss=0.0579 | val_dice=0.9015 | val_iou=0.8610
Epoch 21 step  0.25 | lr=0.001000 | train_loss=0.0383 | train_dice=0.9300 | val_loss=0.0536 | val_dice=0.9107 | val_iou=0.8703
Epoch 21 step  0.50 | lr=0.001000 | train_loss=0.0366 | train_dice=0.9322 | val_loss=0.0554 | val_dice=0.9052 | val_iou=0.8669
Epoch 21 step  0.75 | lr=0.001000 | train_loss=0.0363 | train_dice=0.9327 | val_loss=0.0551 | val_dice=0.9114 | val_iou=0.8706
Új legjobb modell mentve: epoch 21, Dice=0.9114, IoU=0.8706
Epoch 21 step  1.00 | lr=0.001000 | train_loss=0.0361 | train_dice=0.9328 | val_loss=0.0523 | val_dice=0.9141 | val_iou=0.8748
Új legjobb modell mentve: epoch 21, Dice=0.9141, IoU=0.8748
Epoch 21 step  1.00 | lr=0.001000 | train_loss=0.0361 | train_dice=0.9328 | val_loss=0.0523 | val_dice=0.9139 | val_iou=0.8746
Epoch 22 step  0.25 | lr=0.001000 | train_loss=0.0303 | train_dice=0.9436 | val_loss=0.0544 | val_dice=0.9098 | val_iou=0.8711
Epoch 22 step  0.50 | lr=0.001000 | train_loss=0.0315 | train_dice=0.9430 | val_loss=0.0559 | val_dice=0.9057 | val_iou=0.8662
Epoch 22 step  0.75 | lr=0.001000 | train_loss=0.0325 | train_dice=0.9407 | val_loss=0.0536 | val_dice=0.9104 | val_iou=0.8717
Epoch 22 step  1.00 | lr=0.001000 | train_loss=0.0342 | train_dice=0.9372 | val_loss=0.0571 | val_dice=0.9031 | val_iou=0.8641
Epoch 22 step  1.00 | lr=0.001000 | train_loss=0.0342 | train_dice=0.9371 | val_loss=0.0571 | val_dice=0.9030 | val_iou=0.8642
Epoch 23 step  0.25 | lr=0.001000 | train_loss=0.0328 | train_dice=0.9409 | val_loss=0.0510 | val_dice=0.9146 | val_iou=0.8753
Új legjobb modell mentve: epoch 23, Dice=0.9146, IoU=0.8753
Epoch 23 step  0.50 | lr=0.001000 | train_loss=0.0345 | train_dice=0.9380 | val_loss=0.0566 | val_dice=0.9082 | val_iou=0.8695
Epoch 23 step  0.75 | lr=0.001000 | train_loss=0.0339 | train_dice=0.9385 | val_loss=0.0542 | val_dice=0.9092 | val_iou=0.8714
Epoch 23 step  1.00 | lr=0.001000 | train_loss=0.0329 | train_dice=0.9394 | val_loss=0.0548 | val_dice=0.9098 | val_iou=0.8679
Epoch 23 step  1.00 | lr=0.001000 | train_loss=0.0329 | train_dice=0.9395 | val_loss=0.0547 | val_dice=0.9104 | val_iou=0.8690
Epoch 24 step  0.25 | lr=0.001000 | train_loss=0.0338 | train_dice=0.9364 | val_loss=0.0584 | val_dice=0.9048 | val_iou=0.8657
Epoch 24 step  0.50 | lr=0.001000 | train_loss=0.0343 | train_dice=0.9359 | val_loss=0.0518 | val_dice=0.9133 | val_iou=0.8739
Epoch 24 step  0.75 | lr=0.001000 | train_loss=0.0340 | train_dice=0.9374 | val_loss=0.0509 | val_dice=0.9131 | val_iou=0.8735
Epoch 24 step  1.00 | lr=0.001000 | train_loss=0.0337 | train_dice=0.9385 | val_loss=0.0584 | val_dice=0.9042 | val_iou=0.8631
Epoch 24 step  1.00 | lr=0.001000 | train_loss=0.0337 | train_dice=0.9385 | val_loss=0.0572 | val_dice=0.9058 | val_iou=0.8648
Epoch 25 step  0.25 | lr=0.001000 | train_loss=0.0302 | train_dice=0.9445 | val_loss=0.0496 | val_dice=0.9159 | val_iou=0.8773
Új legjobb modell mentve: epoch 25, Dice=0.9159, IoU=0.8773
Epoch 25 step  0.50 | lr=0.001000 | train_loss=0.0316 | train_dice=0.9421 | val_loss=0.0515 | val_dice=0.9142 | val_iou=0.8740
Epoch 25 step  0.75 | lr=0.001000 | train_loss=0.0309 | train_dice=0.9431 | val_loss=0.0510 | val_dice=0.9114 | val_iou=0.8724
Epoch 25 step  1.00 | lr=0.001000 | train_loss=0.0318 | train_dice=0.9417 | val_loss=0.0679 | val_dice=0.8874 | val_iou=0.8458
Epoch 25 step  1.00 | lr=0.001000 | train_loss=0.0319 | train_dice=0.9415 | val_loss=0.0638 | val_dice=0.8936 | val_iou=0.8514
Epoch 26 step  0.25 | lr=0.001000 | train_loss=0.0310 | train_dice=0.9409 | val_loss=0.0476 | val_dice=0.9186 | val_iou=0.8808
Új legjobb modell mentve: epoch 26, Dice=0.9186, IoU=0.8808
Epoch 26 step  0.50 | lr=0.001000 | train_loss=0.0299 | train_dice=0.9433 | val_loss=0.0493 | val_dice=0.9146 | val_iou=0.8757
Epoch 26 step  0.75 | lr=0.001000 | train_loss=0.0300 | train_dice=0.9441 | val_loss=0.0501 | val_dice=0.9150 | val_iou=0.8765
Epoch 26 step  1.00 | lr=0.001000 | train_loss=0.0306 | train_dice=0.9434 | val_loss=0.0541 | val_dice=0.9148 | val_iou=0.8729
Epoch 26 step  1.00 | lr=0.001000 | train_loss=0.0307 | train_dice=0.9431 | val_loss=0.0504 | val_dice=0.9188 | val_iou=0.8788
Új legjobb modell mentve: epoch 26, Dice=0.9188, IoU=0.8788
Epoch 27 step  0.25 | lr=0.001000 | train_loss=0.0274 | train_dice=0.9494 | val_loss=0.0497 | val_dice=0.9183 | val_iou=0.8789
Epoch 27 step  0.50 | lr=0.001000 | train_loss=0.0303 | train_dice=0.9446 | val_loss=0.0512 | val_dice=0.9131 | val_iou=0.8741
Epoch 27 step  0.75 | lr=0.001000 | train_loss=0.0298 | train_dice=0.9452 | val_loss=0.0548 | val_dice=0.9131 | val_iou=0.8721
Epoch 27 step  1.00 | lr=0.001000 | train_loss=0.0294 | train_dice=0.9458 | val_loss=0.0491 | val_dice=0.9174 | val_iou=0.8796
Epoch 27 step  1.00 | lr=0.001000 | train_loss=0.0294 | train_dice=0.9458 | val_loss=0.0486 | val_dice=0.9180 | val_iou=0.8803
Epoch 28 step  0.25 | lr=0.001000 | train_loss=0.0286 | train_dice=0.9482 | val_loss=0.0486 | val_dice=0.9165 | val_iou=0.8785
Epoch 28 step  0.50 | lr=0.001000 | train_loss=0.0290 | train_dice=0.9454 | val_loss=0.0499 | val_dice=0.9167 | val_iou=0.8784
Epoch 28 step  0.75 | lr=0.001000 | train_loss=0.0293 | train_dice=0.9454 | val_loss=0.0490 | val_dice=0.9185 | val_iou=0.8794
Epoch 28 step  1.00 | lr=0.001000 | train_loss=0.0305 | train_dice=0.9444 | val_loss=0.0502 | val_dice=0.9156 | val_iou=0.8770
Epoch 28 step  1.00 | lr=0.001000 | train_loss=0.0305 | train_dice=0.9444 | val_loss=0.0501 | val_dice=0.9165 | val_iou=0.8775
Epoch 29 step  0.25 | lr=0.001000 | train_loss=0.0297 | train_dice=0.9448 | val_loss=0.0489 | val_dice=0.9170 | val_iou=0.8794
Epoch 29 step  0.50 | lr=0.001000 | train_loss=0.0276 | train_dice=0.9489 | val_loss=0.0509 | val_dice=0.9137 | val_iou=0.8745
Epoch 29 step  0.75 | lr=0.001000 | train_loss=0.0281 | train_dice=0.9478 | val_loss=0.0506 | val_dice=0.9147 | val_iou=0.8754
Epoch 29 step  1.00 | lr=0.001000 | train_loss=0.0282 | train_dice=0.9480 | val_loss=0.0489 | val_dice=0.9159 | val_iou=0.8779
Epoch 29 step  1.00 | lr=0.001000 | train_loss=0.0282 | train_dice=0.9480 | val_loss=0.0502 | val_dice=0.9136 | val_iou=0.8757
Epoch 30 step  0.25 | lr=0.001000 | train_loss=0.0258 | train_dice=0.9515 | val_loss=0.0487 | val_dice=0.9160 | val_iou=0.8770
Epoch 30 step  0.50 | lr=0.001000 | train_loss=0.0260 | train_dice=0.9516 | val_loss=0.0501 | val_dice=0.9105 | val_iou=0.8737
Epoch 30 step  0.75 | lr=0.001000 | train_loss=0.0261 | train_dice=0.9516 | val_loss=0.0517 | val_dice=0.9147 | val_iou=0.8775
Epoch 30 step  1.00 | lr=0.001000 | train_loss=0.0265 | train_dice=0.9512 | val_loss=0.0481 | val_dice=0.9203 | val_iou=0.8821
Új legjobb modell mentve: epoch 30, Dice=0.9203, IoU=0.8821
Epoch 30 step  1.00 | lr=0.001000 | train_loss=0.0265 | train_dice=0.9511 | val_loss=0.0481 | val_dice=0.9192 | val_iou=0.8807
Epoch 31 step  0.25 | lr=0.001000 | train_loss=0.0261 | train_dice=0.9503 | val_loss=0.0517 | val_dice=0.9132 | val_iou=0.8747
Epoch 31 step  0.50 | lr=0.001000 | train_loss=0.0261 | train_dice=0.9507 | val_loss=0.0511 | val_dice=0.9147 | val_iou=0.8764
Epoch 31 step  0.75 | lr=0.001000 | train_loss=0.0273 | train_dice=0.9497 | val_loss=0.0524 | val_dice=0.9083 | val_iou=0.8701
Epoch 31 step  1.00 | lr=0.001000 | train_loss=0.0271 | train_dice=0.9501 | val_loss=0.0494 | val_dice=0.9184 | val_iou=0.8798
Epoch 31 step  1.00 | lr=0.001000 | train_loss=0.0271 | train_dice=0.9501 | val_loss=0.0496 | val_dice=0.9176 | val_iou=0.8789
Epoch 32 step  0.25 | lr=0.001000 | train_loss=0.0251 | train_dice=0.9529 | val_loss=0.0491 | val_dice=0.9169 | val_iou=0.8787
Epoch 32 step  0.50 | lr=0.001000 | train_loss=0.0238 | train_dice=0.9556 | val_loss=0.0476 | val_dice=0.9177 | val_iou=0.8798
Epoch 32 step  0.75 | lr=0.001000 | train_loss=0.0242 | train_dice=0.9546 | val_loss=0.0502 | val_dice=0.9144 | val_iou=0.8752
Epoch 32 step  1.00 | lr=0.001000 | train_loss=0.0249 | train_dice=0.9537 | val_loss=0.0496 | val_dice=0.9155 | val_iou=0.8768
Epoch 32 step  1.00 | lr=0.001000 | train_loss=0.0249 | train_dice=0.9537 | val_loss=0.0498 | val_dice=0.9155 | val_iou=0.8772
Epoch 33 step  0.25 | lr=0.001000 | train_loss=0.0245 | train_dice=0.9550 | val_loss=0.0480 | val_dice=0.9165 | val_iou=0.8790
Epoch 33 step  0.50 | lr=0.001000 | train_loss=0.0248 | train_dice=0.9538 | val_loss=0.0481 | val_dice=0.9204 | val_iou=0.8822
Új legjobb modell mentve: epoch 33, Dice=0.9204, IoU=0.8822
Epoch 33 step  0.75 | lr=0.001000 | train_loss=0.0244 | train_dice=0.9542 | val_loss=0.0502 | val_dice=0.9154 | val_iou=0.8765
Epoch 33 step  1.00 | lr=0.001000 | train_loss=0.0265 | train_dice=0.9510 | val_loss=0.0529 | val_dice=0.9137 | val_iou=0.8746
Epoch 33 step  1.00 | lr=0.001000 | train_loss=0.0265 | train_dice=0.9511 | val_loss=0.0525 | val_dice=0.9148 | val_iou=0.8756
Epoch 34 step  0.25 | lr=0.001000 | train_loss=0.0260 | train_dice=0.9515 | val_loss=0.0506 | val_dice=0.9172 | val_iou=0.8783
Epoch 34 step  0.50 | lr=0.001000 | train_loss=0.0249 | train_dice=0.9533 | val_loss=0.0517 | val_dice=0.9166 | val_iou=0.8794
Epoch 34 step  0.75 | lr=0.001000 | train_loss=0.0247 | train_dice=0.9536 | val_loss=0.0549 | val_dice=0.9114 | val_iou=0.8712
Epoch 34 step  1.00 | lr=0.001000 | train_loss=0.0255 | train_dice=0.9526 | val_loss=0.0496 | val_dice=0.9168 | val_iou=0.8786
Epoch 34 step  1.00 | lr=0.001000 | train_loss=0.0255 | train_dice=0.9527 | val_loss=0.0494 | val_dice=0.9169 | val_iou=0.8789
Epoch 35 step  0.25 | lr=0.001000 | train_loss=0.0226 | train_dice=0.9560 | val_loss=0.0488 | val_dice=0.9164 | val_iou=0.8781
Epoch 35 step  0.50 | lr=0.001000 | train_loss=0.0220 | train_dice=0.9583 | val_loss=0.0470 | val_dice=0.9217 | val_iou=0.8837
Új legjobb modell mentve: epoch 35, Dice=0.9217, IoU=0.8837
Epoch 35 step  0.75 | lr=0.001000 | train_loss=0.0225 | train_dice=0.9571 | val_loss=0.0478 | val_dice=0.9209 | val_iou=0.8817
Epoch 35 step  1.00 | lr=0.001000 | train_loss=0.0227 | train_dice=0.9576 | val_loss=0.0495 | val_dice=0.9180 | val_iou=0.8805
Epoch 35 step  1.00 | lr=0.001000 | train_loss=0.0227 | train_dice=0.9576 | val_loss=0.0495 | val_dice=0.9168 | val_iou=0.8794
Epoch 36 step  0.25 | lr=0.001000 | train_loss=0.0210 | train_dice=0.9608 | val_loss=0.0489 | val_dice=0.9167 | val_iou=0.8791
Epoch 36 step  0.50 | lr=0.001000 | train_loss=0.0221 | train_dice=0.9586 | val_loss=0.0492 | val_dice=0.9185 | val_iou=0.8804
Epoch 36 step  0.75 | lr=0.001000 | train_loss=0.0222 | train_dice=0.9588 | val_loss=0.0502 | val_dice=0.9164 | val_iou=0.8781
Epoch 36 step  1.00 | lr=0.001000 | train_loss=0.0228 | train_dice=0.9575 | val_loss=0.0518 | val_dice=0.9162 | val_iou=0.8768
Epoch 36 step  1.00 | lr=0.001000 | train_loss=0.0228 | train_dice=0.9574 | val_loss=0.0523 | val_dice=0.9153 | val_iou=0.8765
Epoch 37 step  0.25 | lr=0.001000 | train_loss=0.0231 | train_dice=0.9579 | val_loss=0.0487 | val_dice=0.9193 | val_iou=0.8813
Epoch 37 step  0.50 | lr=0.001000 | train_loss=0.0219 | train_dice=0.9592 | val_loss=0.0483 | val_dice=0.9204 | val_iou=0.8825
Epoch 37 step  0.75 | lr=0.001000 | train_loss=0.0219 | train_dice=0.9589 | val_loss=0.0508 | val_dice=0.9155 | val_iou=0.8775
Epoch 37 step  1.00 | lr=0.001000 | train_loss=0.0240 | train_dice=0.9558 | val_loss=0.0612 | val_dice=0.9011 | val_iou=0.8610
Epoch 37 step  1.00 | lr=0.001000 | train_loss=0.0241 | train_dice=0.9556 | val_loss=0.0589 | val_dice=0.9054 | val_iou=0.8650
Epoch 38 step  0.25 | lr=0.001000 | train_loss=0.0245 | train_dice=0.9550 | val_loss=0.0491 | val_dice=0.9176 | val_iou=0.8800
Epoch 38 step  0.50 | lr=0.001000 | train_loss=0.0237 | train_dice=0.9568 | val_loss=0.0505 | val_dice=0.9168 | val_iou=0.8777
Epoch 38 step  0.75 | lr=0.001000 | train_loss=0.0228 | train_dice=0.9583 | val_loss=0.0492 | val_dice=0.9198 | val_iou=0.8807
Epoch 38 step  1.00 | lr=0.001000 | train_loss=0.0226 | train_dice=0.9584 | val_loss=0.0512 | val_dice=0.9179 | val_iou=0.8788
Epoch 38 step  1.00 | lr=0.001000 | train_loss=0.0226 | train_dice=0.9585 | val_loss=0.0519 | val_dice=0.9176 | val_iou=0.8787
Epoch 39 step  0.25 | lr=0.001000 | train_loss=0.0211 | train_dice=0.9608 | val_loss=0.0501 | val_dice=0.9148 | val_iou=0.8758
Epoch 39 step  0.50 | lr=0.001000 | train_loss=0.0212 | train_dice=0.9602 | val_loss=0.0512 | val_dice=0.9148 | val_iou=0.8773
Epoch 39 step  0.75 | lr=0.001000 | train_loss=0.0210 | train_dice=0.9607 | val_loss=0.0491 | val_dice=0.9174 | val_iou=0.8803
Epoch 39 step  1.00 | lr=0.001000 | train_loss=0.0211 | train_dice=0.9608 | val_loss=0.0487 | val_dice=0.9214 | val_iou=0.8833
Epoch 39 step  1.00 | lr=0.001000 | train_loss=0.0210 | train_dice=0.9609 | val_loss=0.0487 | val_dice=0.9212 | val_iou=0.8831
Epoch 40 step  0.25 | lr=0.001000 | train_loss=0.0209 | train_dice=0.9598 | val_loss=0.0503 | val_dice=0.9178 | val_iou=0.8803
Epoch 40 step  0.50 | lr=0.001000 | train_loss=0.0201 | train_dice=0.9616 | val_loss=0.0520 | val_dice=0.9169 | val_iou=0.8787
Epoch 40 step  0.75 | lr=0.001000 | train_loss=0.0210 | train_dice=0.9603 | val_loss=0.0525 | val_dice=0.9162 | val_iou=0.8779
Epoch 40 step  1.00 | lr=0.001000 | train_loss=0.0224 | train_dice=0.9587 | val_loss=0.0519 | val_dice=0.9176 | val_iou=0.8795
Epoch 40 step  1.00 | lr=0.001000 | train_loss=0.0225 | train_dice=0.9587 | val_loss=0.0510 | val_dice=0.9181 | val_iou=0.8799
Epoch 41 step  0.25 | lr=0.001000 | train_loss=0.0209 | train_dice=0.9613 | val_loss=0.0512 | val_dice=0.9165 | val_iou=0.8787
Epoch 41 step  0.50 | lr=0.001000 | train_loss=0.0200 | train_dice=0.9628 | val_loss=0.0487 | val_dice=0.9197 | val_iou=0.8817
Epoch 41 step  0.75 | lr=0.001000 | train_loss=0.0202 | train_dice=0.9631 | val_loss=0.0474 | val_dice=0.9209 | val_iou=0.8838
Epoch 41 step  1.00 | lr=0.001000 | train_loss=0.0205 | train_dice=0.9623 | val_loss=0.0495 | val_dice=0.9187 | val_iou=0.8804
Epoch 41 step  1.00 | lr=0.001000 | train_loss=0.0205 | train_dice=0.9623 | val_loss=0.0499 | val_dice=0.9178 | val_iou=0.8796
Epoch 42 step  0.25 | lr=0.001000 | train_loss=0.0196 | train_dice=0.9646 | val_loss=0.0498 | val_dice=0.9188 | val_iou=0.8812
Epoch 42 step  0.50 | lr=0.001000 | train_loss=0.0205 | train_dice=0.9628 | val_loss=0.0516 | val_dice=0.9151 | val_iou=0.8759
Epoch 42 step  0.75 | lr=0.001000 | train_loss=0.0200 | train_dice=0.9638 | val_loss=0.0466 | val_dice=0.9230 | val_iou=0.8857
Új legjobb modell mentve: epoch 42, Dice=0.9230, IoU=0.8857
Epoch 42 step  1.00 | lr=0.001000 | train_loss=0.0199 | train_dice=0.9636 | val_loss=0.0504 | val_dice=0.9199 | val_iou=0.8819
Epoch 42 step  1.00 | lr=0.001000 | train_loss=0.0199 | train_dice=0.9636 | val_loss=0.0514 | val_dice=0.9183 | val_iou=0.8802
Epoch 43 step  0.25 | lr=0.001000 | train_loss=0.0192 | train_dice=0.9664 | val_loss=0.0494 | val_dice=0.9179 | val_iou=0.8805
Epoch 43 step  0.50 | lr=0.001000 | train_loss=0.0196 | train_dice=0.9642 | val_loss=0.0487 | val_dice=0.9212 | val_iou=0.8823
Epoch 43 step  0.75 | lr=0.001000 | train_loss=0.0191 | train_dice=0.9647 | val_loss=0.0494 | val_dice=0.9200 | val_iou=0.8815
Epoch 43 step  1.00 | lr=0.001000 | train_loss=0.0191 | train_dice=0.9647 | val_loss=0.0489 | val_dice=0.9222 | val_iou=0.8835
Epoch 43 step  1.00 | lr=0.001000 | train_loss=0.0191 | train_dice=0.9647 | val_loss=0.0492 | val_dice=0.9218 | val_iou=0.8830
Epoch 44 step  0.25 | lr=0.001000 | train_loss=0.0201 | train_dice=0.9630 | val_loss=0.0493 | val_dice=0.9178 | val_iou=0.8784
Epoch 44 step  0.50 | lr=0.001000 | train_loss=0.0209 | train_dice=0.9604 | val_loss=0.0503 | val_dice=0.9173 | val_iou=0.8794
Epoch 44 step  0.75 | lr=0.001000 | train_loss=0.0211 | train_dice=0.9604 | val_loss=0.0571 | val_dice=0.9099 | val_iou=0.8693
Epoch 44 step  1.00 | lr=0.001000 | train_loss=0.0217 | train_dice=0.9597 | val_loss=0.0501 | val_dice=0.9186 | val_iou=0.8799
Epoch 44 step  1.00 | lr=0.001000 | train_loss=0.0217 | train_dice=0.9596 | val_loss=0.0502 | val_dice=0.9189 | val_iou=0.8802
Epoch 45 step  0.25 | lr=0.001000 | train_loss=0.0199 | train_dice=0.9645 | val_loss=0.0498 | val_dice=0.9182 | val_iou=0.8800
Epoch 45 step  0.50 | lr=0.001000 | train_loss=0.0188 | train_dice=0.9667 | val_loss=0.0478 | val_dice=0.9220 | val_iou=0.8843
Epoch 45 step  0.75 | lr=0.001000 | train_loss=0.0184 | train_dice=0.9668 | val_loss=0.0486 | val_dice=0.9200 | val_iou=0.8814
Epoch 45 step  1.00 | lr=0.001000 | train_loss=0.0187 | train_dice=0.9656 | val_loss=0.0471 | val_dice=0.9228 | val_iou=0.8845
Epoch 45 step  1.00 | lr=0.001000 | train_loss=0.0187 | train_dice=0.9656 | val_loss=0.0472 | val_dice=0.9225 | val_iou=0.8843
Epoch 46 step  0.25 | lr=0.001000 | train_loss=0.0178 | train_dice=0.9662 | val_loss=0.0473 | val_dice=0.9225 | val_iou=0.8838
Epoch 46 step  0.50 | lr=0.001000 | train_loss=0.0174 | train_dice=0.9674 | val_loss=0.0495 | val_dice=0.9199 | val_iou=0.8816
Epoch 46 step  0.75 | lr=0.001000 | train_loss=0.0176 | train_dice=0.9676 | val_loss=0.0472 | val_dice=0.9212 | val_iou=0.8830
Epoch 46 step  1.00 | lr=0.001000 | train_loss=0.0179 | train_dice=0.9667 | val_loss=0.0490 | val_dice=0.9188 | val_iou=0.8804
Epoch 46 step  1.00 | lr=0.001000 | train_loss=0.0179 | train_dice=0.9668 | val_loss=0.0492 | val_dice=0.9187 | val_iou=0.8806
Epoch 47 step  0.25 | lr=0.001000 | train_loss=0.0178 | train_dice=0.9654 | val_loss=0.0483 | val_dice=0.9235 | val_iou=0.8847
Új legjobb modell mentve: epoch 47, Dice=0.9235, IoU=0.8847
Epoch 47 step  0.50 | lr=0.001000 | train_loss=0.0172 | train_dice=0.9671 | val_loss=0.0481 | val_dice=0.9231 | val_iou=0.8842
Epoch 47 step  0.75 | lr=0.001000 | train_loss=0.0174 | train_dice=0.9673 | val_loss=0.0494 | val_dice=0.9218 | val_iou=0.8830
Epoch 47 step  1.00 | lr=0.001000 | train_loss=0.0175 | train_dice=0.9672 | val_loss=0.0531 | val_dice=0.9144 | val_iou=0.8759
Epoch 47 step  1.00 | lr=0.001000 | train_loss=0.0175 | train_dice=0.9671 | val_loss=0.0518 | val_dice=0.9160 | val_iou=0.8781
Epoch 48 step  0.25 | lr=0.001000 | train_loss=0.0170 | train_dice=0.9668 | val_loss=0.0502 | val_dice=0.9179 | val_iou=0.8793
Epoch 48 step  0.50 | lr=0.001000 | train_loss=0.0169 | train_dice=0.9676 | val_loss=0.0502 | val_dice=0.9225 | val_iou=0.8847
Epoch 48 step  0.75 | lr=0.001000 | train_loss=0.0179 | train_dice=0.9663 | val_loss=0.0501 | val_dice=0.9200 | val_iou=0.8816
Epoch 48 step  1.00 | lr=0.001000 | train_loss=0.0180 | train_dice=0.9665 | val_loss=0.0507 | val_dice=0.9168 | val_iou=0.8769
Epoch 48 step  1.00 | lr=0.001000 | train_loss=0.0180 | train_dice=0.9665 | val_loss=0.0504 | val_dice=0.9167 | val_iou=0.8774
Epoch 49 step  0.25 | lr=0.001000 | train_loss=0.0225 | train_dice=0.9605 | val_loss=0.0541 | val_dice=0.9120 | val_iou=0.8725
Epoch 49 step  0.50 | lr=0.001000 | train_loss=0.0210 | train_dice=0.9626 | val_loss=0.0490 | val_dice=0.9200 | val_iou=0.8823
Epoch 49 step  0.75 | lr=0.001000 | train_loss=0.0216 | train_dice=0.9613 | val_loss=0.0495 | val_dice=0.9192 | val_iou=0.8811
Epoch 49 step  1.00 | lr=0.001000 | train_loss=0.0209 | train_dice=0.9623 | val_loss=0.0512 | val_dice=0.9183 | val_iou=0.8806
Epoch 49 step  1.00 | lr=0.001000 | train_loss=0.0209 | train_dice=0.9623 | val_loss=0.0507 | val_dice=0.9190 | val_iou=0.8814
Epoch 50 step  0.25 | lr=0.001000 | train_loss=0.0158 | train_dice=0.9706 | val_loss=0.0476 | val_dice=0.9221 | val_iou=0.8852
Epoch 50 step  0.50 | lr=0.001000 | train_loss=0.0161 | train_dice=0.9699 | val_loss=0.0473 | val_dice=0.9243 | val_iou=0.8866
Új legjobb modell mentve: epoch 50, Dice=0.9243, IoU=0.8866
Epoch 50 step  0.75 | lr=0.001000 | train_loss=0.0160 | train_dice=0.9702 | val_loss=0.0477 | val_dice=0.9236 | val_iou=0.8856
Epoch 50 step  1.00 | lr=0.001000 | train_loss=0.0164 | train_dice=0.9699 | val_loss=0.0481 | val_dice=0.9237 | val_iou=0.8858
Epoch 50 step  1.00 | lr=0.001000 | train_loss=0.0164 | train_dice=0.9698 | val_loss=0.0478 | val_dice=0.9242 | val_iou=0.8863
Epoch 51 step  0.25 | lr=0.001000 | train_loss=0.0153 | train_dice=0.9718 | val_loss=0.0481 | val_dice=0.9239 | val_iou=0.8858
Epoch 51 step  0.50 | lr=0.001000 | train_loss=0.0155 | train_dice=0.9715 | val_loss=0.0492 | val_dice=0.9224 | val_iou=0.8849
Epoch 51 step  0.75 | lr=0.001000 | train_loss=0.0155 | train_dice=0.9712 | val_loss=0.0495 | val_dice=0.9222 | val_iou=0.8844
Epoch 51 step  1.00 | lr=0.001000 | train_loss=0.0164 | train_dice=0.9699 | val_loss=0.0489 | val_dice=0.9197 | val_iou=0.8814
Epoch 51 step  1.00 | lr=0.001000 | train_loss=0.0164 | train_dice=0.9699 | val_loss=0.0486 | val_dice=0.9201 | val_iou=0.8813
Epoch 52 step  0.25 | lr=0.001000 | train_loss=0.0160 | train_dice=0.9697 | val_loss=0.0486 | val_dice=0.9217 | val_iou=0.8834
Epoch 52 step  0.50 | lr=0.001000 | train_loss=0.0155 | train_dice=0.9709 | val_loss=0.0501 | val_dice=0.9206 | val_iou=0.8826
Epoch 52 step  0.75 | lr=0.001000 | train_loss=0.0158 | train_dice=0.9706 | val_loss=0.0495 | val_dice=0.9215 | val_iou=0.8841
Epoch 52 step  1.00 | lr=0.001000 | train_loss=0.0162 | train_dice=0.9700 | val_loss=0.0488 | val_dice=0.9224 | val_iou=0.8846
Epoch 52 step  1.00 | lr=0.001000 | train_loss=0.0162 | train_dice=0.9700 | val_loss=0.0494 | val_dice=0.9214 | val_iou=0.8837
Epoch 53 step  0.25 | lr=0.001000 | train_loss=0.0154 | train_dice=0.9713 | val_loss=0.0497 | val_dice=0.9215 | val_iou=0.8833
Epoch 53 step  0.50 | lr=0.001000 | train_loss=0.0163 | train_dice=0.9697 | val_loss=0.0492 | val_dice=0.9209 | val_iou=0.8828
Epoch 53 step  0.75 | lr=0.001000 | train_loss=0.0162 | train_dice=0.9704 | val_loss=0.0488 | val_dice=0.9224 | val_iou=0.8842
Epoch 53 step  1.00 | lr=0.001000 | train_loss=0.0161 | train_dice=0.9703 | val_loss=0.0508 | val_dice=0.9217 | val_iou=0.8832
Epoch 53 step  1.00 | lr=0.001000 | train_loss=0.0161 | train_dice=0.9703 | val_loss=0.0520 | val_dice=0.9208 | val_iou=0.8823
Epoch 54 step  0.25 | lr=0.001000 | train_loss=0.0180 | train_dice=0.9650 | val_loss=0.0532 | val_dice=0.9164 | val_iou=0.8784
Epoch 54 step  0.50 | lr=0.001000 | train_loss=0.0166 | train_dice=0.9688 | val_loss=0.0502 | val_dice=0.9211 | val_iou=0.8831
Epoch 54 step  0.75 | lr=0.001000 | train_loss=0.0161 | train_dice=0.9704 | val_loss=0.0482 | val_dice=0.9216 | val_iou=0.8828
Epoch 54 step  1.00 | lr=0.001000 | train_loss=0.0163 | train_dice=0.9703 | val_loss=0.0505 | val_dice=0.9182 | val_iou=0.8794
Epoch 54 step  1.00 | lr=0.001000 | train_loss=0.0163 | train_dice=0.9703 | val_loss=0.0504 | val_dice=0.9186 | val_iou=0.8800
Epoch 55 step  0.25 | lr=0.001000 | train_loss=0.0152 | train_dice=0.9711 | val_loss=0.0493 | val_dice=0.9215 | val_iou=0.8839
Epoch 55 step  0.50 | lr=0.001000 | train_loss=0.0152 | train_dice=0.9717 | val_loss=0.0516 | val_dice=0.9185 | val_iou=0.8807
Epoch 55 step  0.75 | lr=0.001000 | train_loss=0.0152 | train_dice=0.9718 | val_loss=0.0495 | val_dice=0.9224 | val_iou=0.8844
Epoch 55 step  1.00 | lr=0.001000 | train_loss=0.0151 | train_dice=0.9723 | val_loss=0.0521 | val_dice=0.9196 | val_iou=0.8819
Epoch 55 step  1.00 | lr=0.001000 | train_loss=0.0151 | train_dice=0.9723 | val_loss=0.0521 | val_dice=0.9192 | val_iou=0.8814
Epoch 56 step  0.25 | lr=0.001000 | train_loss=0.0149 | train_dice=0.9725 | val_loss=0.0514 | val_dice=0.9165 | val_iou=0.8784
Epoch 56 step  0.50 | lr=0.001000 | train_loss=0.0148 | train_dice=0.9734 | val_loss=0.0522 | val_dice=0.9197 | val_iou=0.8817
Epoch 56 step  0.75 | lr=0.001000 | train_loss=0.0158 | train_dice=0.9710 | val_loss=0.0500 | val_dice=0.9212 | val_iou=0.8826
Epoch 56 step  1.00 | lr=0.001000 | train_loss=0.0156 | train_dice=0.9716 | val_loss=0.0492 | val_dice=0.9206 | val_iou=0.8828
Epoch 56 step  1.00 | lr=0.001000 | train_loss=0.0156 | train_dice=0.9716 | val_loss=0.0493 | val_dice=0.9207 | val_iou=0.8829
Epoch 57 step  0.25 | lr=0.001000 | train_loss=0.0140 | train_dice=0.9747 | val_loss=0.0488 | val_dice=0.9237 | val_iou=0.8857
Epoch 57 step  0.50 | lr=0.001000 | train_loss=0.0149 | train_dice=0.9724 | val_loss=0.0540 | val_dice=0.9144 | val_iou=0.8741
Epoch 57 step  0.75 | lr=0.001000 | train_loss=0.0151 | train_dice=0.9718 | val_loss=0.0495 | val_dice=0.9238 | val_iou=0.8860
Epoch 57 step  1.00 | lr=0.001000 | train_loss=0.0149 | train_dice=0.9724 | val_loss=0.0505 | val_dice=0.9225 | val_iou=0.8846
Epoch 57 step  1.00 | lr=0.001000 | train_loss=0.0150 | train_dice=0.9724 | val_loss=0.0497 | val_dice=0.9235 | val_iou=0.8856
Epoch 58 step  0.25 | lr=0.001000 | train_loss=0.0146 | train_dice=0.9707 | val_loss=0.0522 | val_dice=0.9203 | val_iou=0.8813
Epoch 58 step  0.50 | lr=0.001000 | train_loss=0.0147 | train_dice=0.9721 | val_loss=0.0502 | val_dice=0.9222 | val_iou=0.8849
Epoch 58 step  0.75 | lr=0.001000 | train_loss=0.0147 | train_dice=0.9725 | val_loss=0.0495 | val_dice=0.9225 | val_iou=0.8848
Epoch 58 step  1.00 | lr=0.001000 | train_loss=0.0146 | train_dice=0.9730 | val_loss=0.0495 | val_dice=0.9235 | val_iou=0.8858
Epoch 58 step  1.00 | lr=0.001000 | train_loss=0.0146 | train_dice=0.9730 | val_loss=0.0492 | val_dice=0.9237 | val_iou=0.8859
Epoch 59 step  0.25 | lr=0.001000 | train_loss=0.0141 | train_dice=0.9734 | val_loss=0.0501 | val_dice=0.9229 | val_iou=0.8845
Epoch 59 step  0.50 | lr=0.001000 | train_loss=0.0141 | train_dice=0.9737 | val_loss=0.0495 | val_dice=0.9231 | val_iou=0.8853
Epoch 59 step  0.75 | lr=0.001000 | train_loss=0.0139 | train_dice=0.9744 | val_loss=0.0493 | val_dice=0.9238 | val_iou=0.8859
Epoch 59 step  1.00 | lr=0.001000 | train_loss=0.0138 | train_dice=0.9745 | val_loss=0.0501 | val_dice=0.9219 | val_iou=0.8831
Epoch 59 step  1.00 | lr=0.001000 | train_loss=0.0138 | train_dice=0.9745 | val_loss=0.0498 | val_dice=0.9222 | val_iou=0.8836
Epoch 60 step  0.25 | lr=0.001000 | train_loss=0.0172 | train_dice=0.9683 | val_loss=0.0530 | val_dice=0.9181 | val_iou=0.8801
Epoch 60 step  0.50 | lr=0.001000 | train_loss=0.0165 | train_dice=0.9700 | val_loss=0.0511 | val_dice=0.9219 | val_iou=0.8845
Epoch 60 step  0.75 | lr=0.001000 | train_loss=0.0159 | train_dice=0.9709 | val_loss=0.0487 | val_dice=0.9220 | val_iou=0.8839
Epoch 60 step  1.00 | lr=0.001000 | train_loss=0.0153 | train_dice=0.9720 | val_loss=0.0482 | val_dice=0.9230 | val_iou=0.8859
Epoch 60 step  1.00 | lr=0.001000 | train_loss=0.0153 | train_dice=0.9720 | val_loss=0.0484 | val_dice=0.9230 | val_iou=0.8859
Epoch 61 step  0.25 | lr=0.001000 | train_loss=0.0134 | train_dice=0.9754 | val_loss=0.0487 | val_dice=0.9238 | val_iou=0.8863
Epoch 61 step  0.50 | lr=0.001000 | train_loss=0.0134 | train_dice=0.9751 | val_loss=0.0502 | val_dice=0.9232 | val_iou=0.8860
Epoch 61 step  0.75 | lr=0.001000 | train_loss=0.0138 | train_dice=0.9745 | val_loss=0.0507 | val_dice=0.9203 | val_iou=0.8812
Epoch 61 step  1.00 | lr=0.001000 | train_loss=0.0138 | train_dice=0.9747 | val_loss=0.0510 | val_dice=0.9214 | val_iou=0.8834
Epoch 61 step  1.00 | lr=0.001000 | train_loss=0.0138 | train_dice=0.9747 | val_loss=0.0510 | val_dice=0.9210 | val_iou=0.8826
Epoch 62 step  0.25 | lr=0.001000 | train_loss=0.0142 | train_dice=0.9728 | val_loss=0.0498 | val_dice=0.9225 | val_iou=0.8845
Epoch 62 step  0.50 | lr=0.001000 | train_loss=0.0136 | train_dice=0.9744 | val_loss=0.0511 | val_dice=0.9207 | val_iou=0.8831
Epoch 62 step  0.75 | lr=0.001000 | train_loss=0.0143 | train_dice=0.9734 | val_loss=0.0508 | val_dice=0.9222 | val_iou=0.8840
Epoch 62 step  1.00 | lr=0.001000 | train_loss=0.0142 | train_dice=0.9736 | val_loss=0.0494 | val_dice=0.9243 | val_iou=0.8869
Epoch 62 step  1.00 | lr=0.001000 | train_loss=0.0142 | train_dice=0.9736 | val_loss=0.0494 | val_dice=0.9242 | val_iou=0.8869
Epoch 63 step  0.25 | lr=0.001000 | train_loss=0.0135 | train_dice=0.9755 | val_loss=0.0511 | val_dice=0.9213 | val_iou=0.8840
Epoch 63 step  0.50 | lr=0.001000 | train_loss=0.0134 | train_dice=0.9759 | val_loss=0.0499 | val_dice=0.9215 | val_iou=0.8837
Epoch 63 step  0.75 | lr=0.001000 | train_loss=0.0130 | train_dice=0.9761 | val_loss=0.0486 | val_dice=0.9244 | val_iou=0.8864
Új legjobb modell mentve: epoch 63, Dice=0.9244, IoU=0.8864
Epoch 63 step  1.00 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9764 | val_loss=0.0500 | val_dice=0.9221 | val_iou=0.8844
Epoch 63 step  1.00 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9764 | val_loss=0.0504 | val_dice=0.9216 | val_iou=0.8839
Epoch 64 step  0.25 | lr=0.001000 | train_loss=0.0131 | train_dice=0.9760 | val_loss=0.0510 | val_dice=0.9208 | val_iou=0.8820
Epoch 64 step  0.50 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9760 | val_loss=0.0495 | val_dice=0.9226 | val_iou=0.8849
Epoch 64 step  0.75 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9762 | val_loss=0.0509 | val_dice=0.9224 | val_iou=0.8847
Epoch 64 step  1.00 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9762 | val_loss=0.0524 | val_dice=0.9210 | val_iou=0.8831
Epoch 64 step  1.00 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9762 | val_loss=0.0525 | val_dice=0.9209 | val_iou=0.8832
Epoch 65 step  0.25 | lr=0.001000 | train_loss=0.0153 | train_dice=0.9720 | val_loss=0.0528 | val_dice=0.9166 | val_iou=0.8790
Epoch 65 step  0.50 | lr=0.001000 | train_loss=0.0174 | train_dice=0.9691 | val_loss=0.0504 | val_dice=0.9205 | val_iou=0.8823
Epoch 65 step  0.75 | lr=0.001000 | train_loss=0.0166 | train_dice=0.9702 | val_loss=0.0488 | val_dice=0.9235 | val_iou=0.8857
Epoch 65 step  1.00 | lr=0.001000 | train_loss=0.0157 | train_dice=0.9718 | val_loss=0.0496 | val_dice=0.9239 | val_iou=0.8864
Epoch 65 step  1.00 | lr=0.001000 | train_loss=0.0157 | train_dice=0.9718 | val_loss=0.0489 | val_dice=0.9245 | val_iou=0.8869
Új legjobb modell mentve: epoch 65, Dice=0.9245, IoU=0.8869
Epoch 66 step  0.25 | lr=0.001000 | train_loss=0.0129 | train_dice=0.9764 | val_loss=0.0492 | val_dice=0.9241 | val_iou=0.8862
Epoch 66 step  0.50 | lr=0.001000 | train_loss=0.0125 | train_dice=0.9774 | val_loss=0.0486 | val_dice=0.9246 | val_iou=0.8868
Új legjobb modell mentve: epoch 66, Dice=0.9246, IoU=0.8868
Epoch 66 step  0.75 | lr=0.001000 | train_loss=0.0122 | train_dice=0.9777 | val_loss=0.0481 | val_dice=0.9252 | val_iou=0.8875
Új legjobb modell mentve: epoch 66, Dice=0.9252, IoU=0.8875
Epoch 66 step  1.00 | lr=0.001000 | train_loss=0.0139 | train_dice=0.9748 | val_loss=0.0525 | val_dice=0.9201 | val_iou=0.8798
Epoch 66 step  1.00 | lr=0.001000 | train_loss=0.0139 | train_dice=0.9748 | val_loss=0.0521 | val_dice=0.9205 | val_iou=0.8807
Epoch 67 step  0.25 | lr=0.001000 | train_loss=0.0133 | train_dice=0.9748 | val_loss=0.0483 | val_dice=0.9249 | val_iou=0.8870
Epoch 67 step  0.50 | lr=0.001000 | train_loss=0.0146 | train_dice=0.9735 | val_loss=0.0508 | val_dice=0.9211 | val_iou=0.8831
Epoch 67 step  0.75 | lr=0.001000 | train_loss=0.0140 | train_dice=0.9747 | val_loss=0.0495 | val_dice=0.9240 | val_iou=0.8861
Epoch 67 step  1.00 | lr=0.001000 | train_loss=0.0135 | train_dice=0.9754 | val_loss=0.0492 | val_dice=0.9238 | val_iou=0.8861
Epoch 67 step  1.00 | lr=0.001000 | train_loss=0.0135 | train_dice=0.9754 | val_loss=0.0492 | val_dice=0.9237 | val_iou=0.8859
Epoch 68 step  0.25 | lr=0.001000 | train_loss=0.0121 | train_dice=0.9776 | val_loss=0.0510 | val_dice=0.9214 | val_iou=0.8831
Epoch 68 step  0.50 | lr=0.001000 | train_loss=0.0120 | train_dice=0.9778 | val_loss=0.0500 | val_dice=0.9237 | val_iou=0.8861
Epoch 68 step  0.75 | lr=0.001000 | train_loss=0.0118 | train_dice=0.9781 | val_loss=0.0501 | val_dice=0.9240 | val_iou=0.8865
Epoch 68 step  1.00 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9782 | val_loss=0.0494 | val_dice=0.9232 | val_iou=0.8856
Epoch 68 step  1.00 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9782 | val_loss=0.0488 | val_dice=0.9236 | val_iou=0.8859
Epoch 69 step  0.25 | lr=0.001000 | train_loss=0.0126 | train_dice=0.9772 | val_loss=0.0506 | val_dice=0.9205 | val_iou=0.8828
Epoch 69 step  0.50 | lr=0.001000 | train_loss=0.0125 | train_dice=0.9772 | val_loss=0.0512 | val_dice=0.9215 | val_iou=0.8844
Epoch 69 step  0.75 | lr=0.001000 | train_loss=0.0122 | train_dice=0.9775 | val_loss=0.0499 | val_dice=0.9230 | val_iou=0.8851
Epoch 69 step  1.00 | lr=0.001000 | train_loss=0.0121 | train_dice=0.9779 | val_loss=0.0516 | val_dice=0.9221 | val_iou=0.8848
Epoch 69 step  1.00 | lr=0.001000 | train_loss=0.0121 | train_dice=0.9779 | val_loss=0.0512 | val_dice=0.9223 | val_iou=0.8849
Epoch 70 step  0.25 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9778 | val_loss=0.0521 | val_dice=0.9221 | val_iou=0.8847
Epoch 70 step  0.50 | lr=0.001000 | train_loss=0.0114 | train_dice=0.9787 | val_loss=0.0527 | val_dice=0.9218 | val_iou=0.8843
Epoch 70 step  0.75 | lr=0.001000 | train_loss=0.0115 | train_dice=0.9788 | val_loss=0.0508 | val_dice=0.9233 | val_iou=0.8853
Epoch 70 step  1.00 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9786 | val_loss=0.0538 | val_dice=0.9209 | val_iou=0.8835
Epoch 70 step  1.00 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9786 | val_loss=0.0538 | val_dice=0.9207 | val_iou=0.8831
Epoch 71 step  0.25 | lr=0.001000 | train_loss=0.0137 | train_dice=0.9753 | val_loss=0.0515 | val_dice=0.9223 | val_iou=0.8844
Epoch 71 step  0.50 | lr=0.001000 | train_loss=0.0132 | train_dice=0.9757 | val_loss=0.0505 | val_dice=0.9237 | val_iou=0.8863
Epoch 71 step  0.75 | lr=0.001000 | train_loss=0.0131 | train_dice=0.9760 | val_loss=0.0511 | val_dice=0.9217 | val_iou=0.8842
Epoch 71 step  1.00 | lr=0.001000 | train_loss=0.0128 | train_dice=0.9766 | val_loss=0.0523 | val_dice=0.9218 | val_iou=0.8840
Epoch 71 step  1.00 | lr=0.001000 | train_loss=0.0128 | train_dice=0.9766 | val_loss=0.0522 | val_dice=0.9219 | val_iou=0.8840
Epoch 72 step  0.25 | lr=0.001000 | train_loss=0.0122 | train_dice=0.9780 | val_loss=0.0538 | val_dice=0.9181 | val_iou=0.8809
Epoch 72 step  0.50 | lr=0.001000 | train_loss=0.0118 | train_dice=0.9783 | val_loss=0.0523 | val_dice=0.9222 | val_iou=0.8847
Epoch 72 step  0.75 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9786 | val_loss=0.0501 | val_dice=0.9232 | val_iou=0.8853
Epoch 72 step  1.00 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9787 | val_loss=0.0528 | val_dice=0.9207 | val_iou=0.8839
Epoch 72 step  1.00 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9787 | val_loss=0.0521 | val_dice=0.9214 | val_iou=0.8845
Epoch 73 step  0.25 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9780 | val_loss=0.0507 | val_dice=0.9225 | val_iou=0.8847
Epoch 73 step  0.50 | lr=0.001000 | train_loss=0.0112 | train_dice=0.9792 | val_loss=0.0519 | val_dice=0.9224 | val_iou=0.8851
Epoch 73 step  0.75 | lr=0.001000 | train_loss=0.0114 | train_dice=0.9792 | val_loss=0.0541 | val_dice=0.9201 | val_iou=0.8826
Epoch 73 step  1.00 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9786 | val_loss=0.0527 | val_dice=0.9222 | val_iou=0.8840
Epoch 73 step  1.00 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9786 | val_loss=0.0523 | val_dice=0.9225 | val_iou=0.8841
Epoch 74 step  0.25 | lr=0.001000 | train_loss=0.0120 | train_dice=0.9783 | val_loss=0.0508 | val_dice=0.9195 | val_iou=0.8817
Epoch 74 step  0.50 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9790 | val_loss=0.0516 | val_dice=0.9229 | val_iou=0.8853
Epoch 74 step  0.75 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9788 | val_loss=0.0508 | val_dice=0.9217 | val_iou=0.8840
Epoch 74 step  1.00 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9786 | val_loss=0.0518 | val_dice=0.9220 | val_iou=0.8835
Epoch 74 step  1.00 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9786 | val_loss=0.0516 | val_dice=0.9228 | val_iou=0.8847
Epoch 75 step  0.25 | lr=0.001000 | train_loss=0.0118 | train_dice=0.9785 | val_loss=0.0515 | val_dice=0.9217 | val_iou=0.8827
Epoch 75 step  0.50 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9785 | val_loss=0.0507 | val_dice=0.9230 | val_iou=0.8847
Epoch 75 step  0.75 | lr=0.001000 | train_loss=0.0117 | train_dice=0.9783 | val_loss=0.0504 | val_dice=0.9234 | val_iou=0.8855
Epoch 75 step  1.00 | lr=0.001000 | train_loss=0.0115 | train_dice=0.9787 | val_loss=0.0529 | val_dice=0.9202 | val_iou=0.8826
Epoch 75 step  1.00 | lr=0.001000 | train_loss=0.0115 | train_dice=0.9787 | val_loss=0.0515 | val_dice=0.9227 | val_iou=0.8852
Epoch 76 step  0.25 | lr=0.001000 | train_loss=0.0131 | train_dice=0.9752 | val_loss=0.0526 | val_dice=0.9203 | val_iou=0.8828
Epoch 76 step  0.50 | lr=0.001000 | train_loss=0.0124 | train_dice=0.9771 | val_loss=0.0521 | val_dice=0.9223 | val_iou=0.8846
Epoch 76 step  0.75 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9783 | val_loss=0.0516 | val_dice=0.9211 | val_iou=0.8837
Epoch 76 step  1.00 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9789 | val_loss=0.0528 | val_dice=0.9204 | val_iou=0.8829
Epoch 76 step  1.00 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9788 | val_loss=0.0520 | val_dice=0.9209 | val_iou=0.8834
Epoch 77 step  0.25 | lr=0.001000 | train_loss=0.0111 | train_dice=0.9793 | val_loss=0.0536 | val_dice=0.9173 | val_iou=0.8791
Epoch 77 step  0.50 | lr=0.001000 | train_loss=0.0121 | train_dice=0.9776 | val_loss=0.0511 | val_dice=0.9215 | val_iou=0.8840
Epoch 77 step  0.75 | lr=0.001000 | train_loss=0.0123 | train_dice=0.9775 | val_loss=0.0579 | val_dice=0.9147 | val_iou=0.8748
Epoch 77 step  1.00 | lr=0.001000 | train_loss=0.0127 | train_dice=0.9769 | val_loss=0.0526 | val_dice=0.9217 | val_iou=0.8841
Epoch 77 step  1.00 | lr=0.001000 | train_loss=0.0127 | train_dice=0.9768 | val_loss=0.0524 | val_dice=0.9219 | val_iou=0.8844
Epoch 78 step  0.25 | lr=0.001000 | train_loss=0.0116 | train_dice=0.9780 | val_loss=0.0515 | val_dice=0.9226 | val_iou=0.8846
Epoch 78 step  0.50 | lr=0.001000 | train_loss=0.0112 | train_dice=0.9794 | val_loss=0.0506 | val_dice=0.9244 | val_iou=0.8873
Epoch 78 step  0.75 | lr=0.001000 | train_loss=0.0110 | train_dice=0.9799 | val_loss=0.0502 | val_dice=0.9237 | val_iou=0.8864
Epoch 78 step  1.00 | lr=0.001000 | train_loss=0.0107 | train_dice=0.9803 | val_loss=0.0507 | val_dice=0.9240 | val_iou=0.8868
Epoch 78 step  1.00 | lr=0.001000 | train_loss=0.0107 | train_dice=0.9802 | val_loss=0.0507 | val_dice=0.9240 | val_iou=0.8869
Epoch 79 step  0.25 | lr=0.001000 | train_loss=0.0105 | train_dice=0.9811 | val_loss=0.0506 | val_dice=0.9240 | val_iou=0.8863
Epoch 79 step  0.50 | lr=0.001000 | train_loss=0.0103 | train_dice=0.9815 | val_loss=0.0514 | val_dice=0.9233 | val_iou=0.8863
Epoch 79 step  0.75 | lr=0.001000 | train_loss=0.0101 | train_dice=0.9817 | val_loss=0.0509 | val_dice=0.9247 | val_iou=0.8880
Epoch 79 step  1.00 | lr=0.001000 | train_loss=0.0101 | train_dice=0.9817 | val_loss=0.0505 | val_dice=0.9240 | val_iou=0.8869
Epoch 79 step  1.00 | lr=0.001000 | train_loss=0.0101 | train_dice=0.9817 | val_loss=0.0506 | val_dice=0.9239 | val_iou=0.8869
Epoch 80 step  0.25 | lr=0.001000 | train_loss=0.0098 | train_dice=0.9819 | val_loss=0.0498 | val_dice=0.9253 | val_iou=0.8877
Új legjobb modell mentve: epoch 80, Dice=0.9253, IoU=0.8877
Epoch 80 step  0.50 | lr=0.001000 | train_loss=0.0100 | train_dice=0.9818 | val_loss=0.0512 | val_dice=0.9240 | val_iou=0.8865
Epoch 80 step  0.75 | lr=0.001000 | train_loss=0.0099 | train_dice=0.9821 | val_loss=0.0516 | val_dice=0.9237 | val_iou=0.8868
Epoch 80 step  1.00 | lr=0.001000 | train_loss=0.0100 | train_dice=0.9819 | val_loss=0.0518 | val_dice=0.9228 | val_iou=0.8852
Epoch 80 step  1.00 | lr=0.001000 | train_loss=0.0100 | train_dice=0.9819 | val_loss=0.0521 | val_dice=0.9226 | val_iou=0.8850
Epoch 81 step  0.25 | lr=0.001000 | train_loss=0.0106 | train_dice=0.9805 | val_loss=0.0546 | val_dice=0.9178 | val_iou=0.8804
Epoch 81 step  0.50 | lr=0.001000 | train_loss=0.0106 | train_dice=0.9807 | val_loss=0.0545 | val_dice=0.9208 | val_iou=0.8832
Epoch 81 step  0.75 | lr=0.001000 | train_loss=0.0106 | train_dice=0.9808 | val_loss=0.0525 | val_dice=0.9228 | val_iou=0.8855
Epoch 81 step  1.00 | lr=0.001000 | train_loss=0.0105 | train_dice=0.9810 | val_loss=0.0515 | val_dice=0.9242 | val_iou=0.8867
Epoch 81 step  1.00 | lr=0.001000 | train_loss=0.0105 | train_dice=0.9810 | val_loss=0.0516 | val_dice=0.9242 | val_iou=0.8867
Epoch 82 step  0.25 | lr=0.001000 | train_loss=0.0102 | train_dice=0.9808 | val_loss=0.0512 | val_dice=0.9233 | val_iou=0.8859
Epoch 82 step  0.50 | lr=0.001000 | train_loss=0.0102 | train_dice=0.9813 | val_loss=0.0555 | val_dice=0.9198 | val_iou=0.8825
Epoch 82 step  0.75 | lr=0.001000 | train_loss=0.0101 | train_dice=0.9814 | val_loss=0.0536 | val_dice=0.9227 | val_iou=0.8853
Epoch 82 step  1.00 | lr=0.001000 | train_loss=0.0104 | train_dice=0.9811 | val_loss=0.0526 | val_dice=0.9233 | val_iou=0.8854
Epoch 82 step  1.00 | lr=0.001000 | train_loss=0.0104 | train_dice=0.9811 | val_loss=0.0530 | val_dice=0.9230 | val_iou=0.8852
Epoch 83 step  0.25 | lr=0.001000 | train_loss=0.0118 | train_dice=0.9792 | val_loss=0.0524 | val_dice=0.9194 | val_iou=0.8809
Epoch 83 step  0.50 | lr=0.001000 | train_loss=0.0126 | train_dice=0.9781 | val_loss=0.0543 | val_dice=0.9202 | val_iou=0.8807
Epoch 83 step  0.75 | lr=0.001000 | train_loss=0.0123 | train_dice=0.9781 | val_loss=0.0514 | val_dice=0.9229 | val_iou=0.8851
Epoch 83 step  1.00 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9789 | val_loss=0.0513 | val_dice=0.9238 | val_iou=0.8865
Epoch 83 step  1.00 | lr=0.001000 | train_loss=0.0119 | train_dice=0.9789 | val_loss=0.0513 | val_dice=0.9237 | val_iou=0.8864
Epoch 84 step  0.25 | lr=0.001000 | train_loss=0.0100 | train_dice=0.9817 | val_loss=0.0516 | val_dice=0.9231 | val_iou=0.8858
Epoch 84 step  0.50 | lr=0.001000 | train_loss=0.0098 | train_dice=0.9823 | val_loss=0.0523 | val_dice=0.9237 | val_iou=0.8866
Epoch 84 step  0.75 | lr=0.001000 | train_loss=0.0098 | train_dice=0.9820 | val_loss=0.0515 | val_dice=0.9227 | val_iou=0.8853
Epoch 84 step  1.00 | lr=0.001000 | train_loss=0.0097 | train_dice=0.9823 | val_loss=0.0519 | val_dice=0.9233 | val_iou=0.8862
Epoch 84 step  1.00 | lr=0.001000 | train_loss=0.0097 | train_dice=0.9823 | val_loss=0.0517 | val_dice=0.9236 | val_iou=0.8864
Epoch 85 step  0.25 | lr=0.001000 | train_loss=0.0095 | train_dice=0.9828 | val_loss=0.0519 | val_dice=0.9229 | val_iou=0.8857
Epoch 85 step  0.50 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9835 | val_loss=0.0526 | val_dice=0.9228 | val_iou=0.8857
Epoch 85 step  0.75 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9834 | val_loss=0.0532 | val_dice=0.9219 | val_iou=0.8848
Epoch 85 step  1.00 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9832 | val_loss=0.0528 | val_dice=0.9215 | val_iou=0.8843
Epoch 85 step  1.00 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9832 | val_loss=0.0531 | val_dice=0.9212 | val_iou=0.8839
Epoch 86 step  0.25 | lr=0.001000 | train_loss=0.0094 | train_dice=0.9832 | val_loss=0.0519 | val_dice=0.9224 | val_iou=0.8848
Epoch 86 step  0.50 | lr=0.001000 | train_loss=0.0095 | train_dice=0.9831 | val_loss=0.0528 | val_dice=0.9228 | val_iou=0.8858
Epoch 86 step  0.75 | lr=0.001000 | train_loss=0.0094 | train_dice=0.9831 | val_loss=0.0540 | val_dice=0.9217 | val_iou=0.8846
Epoch 86 step  1.00 | lr=0.001000 | train_loss=0.0102 | train_dice=0.9815 | val_loss=0.0558 | val_dice=0.9167 | val_iou=0.8785
Epoch 86 step  1.00 | lr=0.001000 | train_loss=0.0102 | train_dice=0.9815 | val_loss=0.0552 | val_dice=0.9175 | val_iou=0.8795
Epoch 87 step  0.25 | lr=0.001000 | train_loss=0.0107 | train_dice=0.9803 | val_loss=0.0521 | val_dice=0.9203 | val_iou=0.8836
Epoch 87 step  0.50 | lr=0.001000 | train_loss=0.0104 | train_dice=0.9808 | val_loss=0.0518 | val_dice=0.9227 | val_iou=0.8857
Epoch 87 step  0.75 | lr=0.001000 | train_loss=0.0103 | train_dice=0.9812 | val_loss=0.0523 | val_dice=0.9219 | val_iou=0.8844
Epoch 87 step  1.00 | lr=0.001000 | train_loss=0.0101 | train_dice=0.9816 | val_loss=0.0520 | val_dice=0.9218 | val_iou=0.8847
Epoch 87 step  1.00 | lr=0.001000 | train_loss=0.0101 | train_dice=0.9816 | val_loss=0.0519 | val_dice=0.9219 | val_iou=0.8848
Epoch 88 step  0.25 | lr=0.001000 | train_loss=0.0095 | train_dice=0.9831 | val_loss=0.0529 | val_dice=0.9219 | val_iou=0.8849
Epoch 88 step  0.50 | lr=0.001000 | train_loss=0.0094 | train_dice=0.9832 | val_loss=0.0530 | val_dice=0.9215 | val_iou=0.8843
Epoch 88 step  0.75 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9832 | val_loss=0.0530 | val_dice=0.9219 | val_iou=0.8846
Epoch 88 step  1.00 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9833 | val_loss=0.0521 | val_dice=0.9229 | val_iou=0.8857
Epoch 88 step  1.00 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9833 | val_loss=0.0528 | val_dice=0.9221 | val_iou=0.8849
Epoch 89 step  0.25 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9830 | val_loss=0.0516 | val_dice=0.9231 | val_iou=0.8854
Epoch 89 step  0.50 | lr=0.001000 | train_loss=0.0091 | train_dice=0.9836 | val_loss=0.0557 | val_dice=0.9219 | val_iou=0.8845
Epoch 89 step  0.75 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9835 | val_loss=0.0521 | val_dice=0.9234 | val_iou=0.8855
Epoch 89 step  1.00 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9836 | val_loss=0.0530 | val_dice=0.9232 | val_iou=0.8860
Epoch 89 step  1.00 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9836 | val_loss=0.0532 | val_dice=0.9229 | val_iou=0.8856
Epoch 90 step  0.25 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9833 | val_loss=0.0531 | val_dice=0.9234 | val_iou=0.8857
Epoch 90 step  0.50 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9835 | val_loss=0.0524 | val_dice=0.9226 | val_iou=0.8852
Epoch 90 step  0.75 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9835 | val_loss=0.0531 | val_dice=0.9223 | val_iou=0.8855
Epoch 90 step  1.00 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9836 | val_loss=0.0521 | val_dice=0.9225 | val_iou=0.8852
Epoch 90 step  1.00 | lr=0.001000 | train_loss=0.0092 | train_dice=0.9835 | val_loss=0.0515 | val_dice=0.9230 | val_iou=0.8856
Epoch 91 step  0.25 | lr=0.001000 | train_loss=0.0094 | train_dice=0.9827 | val_loss=0.0534 | val_dice=0.9205 | val_iou=0.8828
Epoch 91 step  0.50 | lr=0.001000 | train_loss=0.0095 | train_dice=0.9829 | val_loss=0.0506 | val_dice=0.9235 | val_iou=0.8856
Epoch 91 step  0.75 | lr=0.001000 | train_loss=0.0094 | train_dice=0.9831 | val_loss=0.0514 | val_dice=0.9227 | val_iou=0.8851
Epoch 91 step  1.00 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9832 | val_loss=0.0518 | val_dice=0.9224 | val_iou=0.8842
Epoch 91 step  1.00 | lr=0.001000 | train_loss=0.0093 | train_dice=0.9832 | val_loss=0.0520 | val_dice=0.9222 | val_iou=0.8842
Epoch 92 step  0.25 | lr=0.001000 | train_loss=0.0097 | train_dice=0.9828 | val_loss=0.0548 | val_dice=0.9209 | val_iou=0.8830
Epoch 92 step  0.50 | lr=0.001000 | train_loss=0.0098 | train_dice=0.9825 | val_loss=0.0531 | val_dice=0.9220 | val_iou=0.8844
Epoch 92 step  0.75 | lr=0.001000 | train_loss=0.0098 | train_dice=0.9827 | val_loss=0.0543 | val_dice=0.9194 | val_iou=0.8821
Epoch 92 step  1.00 | lr=0.001000 | train_loss=0.0099 | train_dice=0.9822 | val_loss=0.0538 | val_dice=0.9185 | val_iou=0.8805
Epoch 92 step  1.00 | lr=0.001000 | train_loss=0.0099 | train_dice=0.9822 | val_loss=0.0540 | val_dice=0.9188 | val_iou=0.8806
Epoch 93 step  0.25 | lr=0.001000 | train_loss=0.0113 | train_dice=0.9808 | val_loss=0.0521 | val_dice=0.9197 | val_iou=0.8813
Epoch 93 step  0.50 | lr=0.001000 | train_loss=0.0104 | train_dice=0.9816 | val_loss=0.0532 | val_dice=0.9203 | val_iou=0.8823
Epoch 93 step  0.75 | lr=0.001000 | train_loss=0.0100 | train_dice=0.9823 | val_loss=0.0512 | val_dice=0.9233 | val_iou=0.8859
Epoch 93 step  1.00 | lr=0.001000 | train_loss=0.0097 | train_dice=0.9828 | val_loss=0.0508 | val_dice=0.9237 | val_iou=0.8866
Epoch 93 step  1.00 | lr=0.001000 | train_loss=0.0097 | train_dice=0.9829 | val_loss=0.0510 | val_dice=0.9236 | val_iou=0.8866
Epoch 94 step  0.25 | lr=0.001000 | train_loss=0.0087 | train_dice=0.9843 | val_loss=0.0518 | val_dice=0.9232 | val_iou=0.8862
Epoch 94 step  0.50 | lr=0.001000 | train_loss=0.0085 | train_dice=0.9847 | val_loss=0.0515 | val_dice=0.9234 | val_iou=0.8860
Epoch 94 step  0.75 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9850 | val_loss=0.0523 | val_dice=0.9239 | val_iou=0.8870
Epoch 94 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9849 | val_loss=0.0520 | val_dice=0.9222 | val_iou=0.8840
Epoch 94 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9849 | val_loss=0.0519 | val_dice=0.9224 | val_iou=0.8844
Epoch 95 step  0.25 | lr=0.001000 | train_loss=0.0086 | train_dice=0.9846 | val_loss=0.0542 | val_dice=0.9215 | val_iou=0.8835
Epoch 95 step  0.50 | lr=0.001000 | train_loss=0.0085 | train_dice=0.9848 | val_loss=0.0544 | val_dice=0.9221 | val_iou=0.8848
Epoch 95 step  0.75 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9849 | val_loss=0.0542 | val_dice=0.9221 | val_iou=0.8843
Epoch 95 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9849 | val_loss=0.0519 | val_dice=0.9244 | val_iou=0.8874
Epoch 95 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9850 | val_loss=0.0520 | val_dice=0.9245 | val_iou=0.8876
Epoch 96 step  0.25 | lr=0.001000 | train_loss=0.0090 | train_dice=0.9842 | val_loss=0.0522 | val_dice=0.9227 | val_iou=0.8855
Epoch 96 step  0.50 | lr=0.001000 | train_loss=0.0088 | train_dice=0.9845 | val_loss=0.0539 | val_dice=0.9213 | val_iou=0.8844
Epoch 96 step  0.75 | lr=0.001000 | train_loss=0.0087 | train_dice=0.9845 | val_loss=0.0523 | val_dice=0.9218 | val_iou=0.8845
Epoch 96 step  1.00 | lr=0.001000 | train_loss=0.0086 | train_dice=0.9846 | val_loss=0.0523 | val_dice=0.9227 | val_iou=0.8850
Epoch 96 step  1.00 | lr=0.001000 | train_loss=0.0086 | train_dice=0.9846 | val_loss=0.0525 | val_dice=0.9227 | val_iou=0.8852
Epoch 97 step  0.25 | lr=0.001000 | train_loss=0.0087 | train_dice=0.9846 | val_loss=0.0536 | val_dice=0.9206 | val_iou=0.8828
Epoch 97 step  0.50 | lr=0.001000 | train_loss=0.0087 | train_dice=0.9844 | val_loss=0.0541 | val_dice=0.9195 | val_iou=0.8812
Epoch 97 step  0.75 | lr=0.001000 | train_loss=0.0088 | train_dice=0.9843 | val_loss=0.0540 | val_dice=0.9221 | val_iou=0.8845
Epoch 97 step  1.00 | lr=0.001000 | train_loss=0.0088 | train_dice=0.9843 | val_loss=0.0536 | val_dice=0.9210 | val_iou=0.8828
Epoch 97 step  1.00 | lr=0.001000 | train_loss=0.0088 | train_dice=0.9843 | val_loss=0.0534 | val_dice=0.9211 | val_iou=0.8829
Epoch 98 step  0.25 | lr=0.001000 | train_loss=0.0087 | train_dice=0.9840 | val_loss=0.0536 | val_dice=0.9212 | val_iou=0.8829
Epoch 98 step  0.50 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9844 | val_loss=0.0545 | val_dice=0.9226 | val_iou=0.8852
Epoch 98 step  0.75 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9846 | val_loss=0.0538 | val_dice=0.9234 | val_iou=0.8861
Epoch 98 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9848 | val_loss=0.0532 | val_dice=0.9230 | val_iou=0.8851
Epoch 98 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9848 | val_loss=0.0533 | val_dice=0.9233 | val_iou=0.8855
Epoch 99 step  0.25 | lr=0.001000 | train_loss=0.0096 | train_dice=0.9820 | val_loss=0.0552 | val_dice=0.9199 | val_iou=0.8805
Epoch 99 step  0.50 | lr=0.001000 | train_loss=0.0094 | train_dice=0.9826 | val_loss=0.0552 | val_dice=0.9208 | val_iou=0.8825
Epoch 99 step  0.75 | lr=0.001000 | train_loss=0.0091 | train_dice=0.9833 | val_loss=0.0543 | val_dice=0.9222 | val_iou=0.8846
Epoch 99 step  1.00 | lr=0.001000 | train_loss=0.0089 | train_dice=0.9838 | val_loss=0.0535 | val_dice=0.9219 | val_iou=0.8844
Epoch 99 step  1.00 | lr=0.001000 | train_loss=0.0089 | train_dice=0.9838 | val_loss=0.0536 | val_dice=0.9218 | val_iou=0.8843
Epoch 100 step  0.25 | lr=0.001000 | train_loss=0.0089 | train_dice=0.9842 | val_loss=0.0561 | val_dice=0.9190 | val_iou=0.8813
Epoch 100 step  0.50 | lr=0.001000 | train_loss=0.0087 | train_dice=0.9844 | val_loss=0.0542 | val_dice=0.9227 | val_iou=0.8857
Epoch 100 step  0.75 | lr=0.001000 | train_loss=0.0085 | train_dice=0.9849 | val_loss=0.0541 | val_dice=0.9225 | val_iou=0.8852
Epoch 100 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9851 | val_loss=0.0549 | val_dice=0.9223 | val_iou=0.8848
Epoch 100 step  1.00 | lr=0.001000 | train_loss=0.0084 | train_dice=0.9852 | val_loss=0.0549 | val_dice=0.9225 | val_iou=0.8849
Best val Dice: 0.9253458717043166
"""

def generate_final_plot(text, save_dir="/content/drive/MyDrive/Brain MRI/Unet_FNO_2/plots/"):
    # Adatok kinyerése
    metrics = {'loss': [], 'val_loss': [], 'dice': [], 'val_dice': [], 'val_iou': []}

    pattern = r"train_loss=([\d\.]+) \| train_dice=([\d\.]+) \| val_loss=([\d\.]+) \| val_dice=([\d\.]+) \| val_iou=([\d\.]+)"
    matches = re.findall(pattern, text)

    for m in matches:
        metrics['loss'].append(float(m[0]))
        metrics['dice'].append(float(m[1]))
        metrics['val_loss'].append(float(m[2]))
        metrics['val_dice'].append(float(m[3]))
        metrics['val_iou'].append(float(m[4]))

    # Tengely beállítása 100 epoch-ra
    total_points = len(metrics['loss'])
    steps = np.linspace(0, 100, total_points)

    # Stílus
    sns.set_style("whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(22, 8))

    # Értékek a címhez
    best_val_dice = max(metrics['val_dice'])
    # Az általad megadott teszt eredmények:
    test_dice = 0.9253
    test_iou = 0.8877

    # Összetett cím beállítása
    main_title = (f"Training History: UNetFNO (run1) | Best Val Dice: {best_val_dice:.4f}\n"
                  f"Test Dice: {test_dice:.4f}, Test IoU: {test_iou:.4f}")

    fig.suptitle(main_title, fontsize=18, fontweight='bold', y=0.98)

    # 1. Loss görbe
    axes[0].plot(steps, metrics['loss'], label='Train Loss', alpha=0.7)
    axes[0].plot(steps, metrics['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_title('Training and Validation Loss', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    # 2. Dice Score görbe
    axes[1].plot(steps, metrics['dice'], label='Train Dice', alpha=0.7)
    axes[1].plot(steps, metrics['val_dice'], label='Val Dice', linewidth=2)
    axes[1].set_title('Dice Score', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].legend()

    # 3. Validation Metrics
    axes[2].plot(steps, metrics['val_dice'], label='Val Dice', color='tab:blue')
    axes[2].plot(steps, metrics['val_iou'], label='Val IoU', color='tab:orange')
    axes[2].set_title('Validation Metrics', fontsize=14)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Score')
    axes[2].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.93])

    # Mentés a DRIVE-ra
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    save_path = os.path.join(save_dir, "UNetFNO_final_report.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Mentve a plots mappába: {save_path}")
    plt.show()

# Futtatás
generate_final_plot(log_text)